Эксперименты по верификации диктора
Сравниваем модели, обработку аудио, обучение и способы принятия решения. Для каждого эксперимента фиксируем гипотезу, протокол, результат и вывод. Train используется для обучения, validation — для выбора вариантов, test — для проверки переноса; ограничения многократного просмотра test указаны отдельно.

In [ ]:
import hashlib
import hmac
import json
import math
import platform
import sys
from pathlib import Path
import csv
import io
import numpy as np
from itertools import combinations
from IPython.display import Markdown, display
from collections import Counter



project_dir = Path(
    r"C:\Users\niki\Desktop\speaker_verification"
).resolve(strict=True)

if not project_dir.is_dir():
    raise NotADirectoryError("Не найден каталог проекта")

print("Python:", sys.version.split()[0])
print("Система:", platform.platform())
print("NumPy:", np.__version__)
print("Проект:", project_dir.name)
print("Обучение, скачивание и загрузка весов не выполнялись.")

Читаем только заранее перечисленные JSON-отчёты с фиксированными SHA-256. Ограничиваем размер чтения, запрещаем повторные ключи и неконечные числа. Хеши фиксируют текущие файлы, но не доказывают достоверность самих экспериментов. Test-отчёты используются для документирования результатов, а не нового подбора параметров.

In [ ]:
reports_manifest = {
    "synthetic_adaptation": (
        "results/fc_adaptation_test.json",
        "37016ce03d07428b0189c5c17c9ea6d56ba364ccbf8836ec43517f7f08fd978f",
    ),
    "wavlm": (
        "results/wavlm_validation.json",
        "32354115f6a5b609df076e57877a181ee0cbc082d30fe0f1e088b47e97239183",
    ),
    "classification_history": (
        "checkpoints/vibravox_fc_classification_1200/history.json",
        "249e55596b9c4838c3757f362eb6bf97f930510d4eee8400d82641055766725a",
    ),
    "contrastive_history": (
        "checkpoints/vibravox_fc_contrastive_1200/history.json",
        "e10b0d7c109d4c61e5990006c44226ecb273aa1eda39a2dd51537e46ea770eec",
    ),
    "test_1000": (
        "data/vibravox/test_1000/evaluation.json",
        "e8c126b5c463eeb0ecb625de6e23d4d02a84f68d7fe109ae1aca0fcef08292ec",
    ),
    "bootstrap": (
        "data/vibravox/test_1000/speaker_bootstrap.json",
        "beaa2647c4c9ad1d2e1c7a2292c39b3823287d4f00bc5346b8940ecd452a71fb",
    ),
    "target_far_test": (
        "data/vibravox/test_1000/target_far_comparison.json",
        "fb89bc289d6f6945912b5d80ead40af86cd9c74c96766a3aa3826e0792ad1adb",
    ),
    "target_far_validation": (
        "data/vibravox/test_1000/validation_operating_candidates.json",
        "f15ee5afb77efb50c4a524df21ece4f14f1e6ddc01b204ef957e55aa69e84b50",
    ),
    "duration": (
        "data/vibravox/duration_experiment/fixed_threshold_results.json",
        "ee0ac75ab17dd03ea55f0076e0b1a4c182e07e9bc12b485ab87e81d61f8ae6e4",
    ),
    "duration_center": (
        "data/vibravox/duration_experiment/test_center_comparison.json",
        "8673a4ae00887f9a23125cc6df0fb3d53e92841b0edb8d820b3f2f700428758e",
    ),
    "pair_classifiers": (
        "experiments/pair_classifiers/validation_comparison.json",
        "138fd99435ce77c0d86c9311882434941fee618448eaafcc627083d217d9acfd",
    ),
    "plda_validation": (
        "experiments/pair_classifiers/plda_validation_comparison.json",
        "c6cbe0d4b2711fbb42c0edc3af542513e3721715d9590d4823716a728a5d9a6a",
    ),
    "plda_test": (
        "experiments/pair_classifiers/plda_test_comparison.json",
        "cfe84f8d32e2184004724ad8187030083753e944a7aa39f46847697c60b3c986",
    ),
    "mfcc": (
        "experiments/mfcc_baseline/validation_comparison.json",
        "b00e3fad01657cebf4dbae88ade5b9c7c69a8d55edb652d89bb7c40f5faad042",
    ),
    "ivector": (
        "experiments/ivector/validation_comparison.json",
        "9a178aabc858634ca774fc8c61b4d1b0eb69c3911ec90d99263856a8d6bff69f",
    ),
    "calibration": (
        "checkpoints/vibravox_fc_classification_1200/"
        "calibration_vibravox_candidate.json",
        "e3f452fbb1eaa8baf2610630885b4a458969251482fa148a959e12ad395c1df4",
    ),
}

max_report_bytes = 256 * 1024


def reject_json_constant(value):
    raise ValueError(f"Недопустимая JSON-константа: {value}")


def finite_json_float(value):
    result = float(value)

    if not math.isfinite(result):
        raise ValueError("JSON содержит неконечное число")

    return result


def unique_json_keys(pairs):
    result = {}

    for key, value in pairs:
        if key in result:
            raise ValueError(f"Повторный JSON-ключ: {key}")

        result[key] = value

    return result


def read_verified_report(name):
    if name not in reports_manifest:
        raise ValueError("Отчёт отсутствует в разрешённом списке")

    relative_path, expected_hash = reports_manifest[name]
    path = (project_dir / relative_path).resolve(strict=True)

    if not path.is_relative_to(project_dir):
        raise ValueError("Отчёт находится вне каталога проекта")

    if not path.is_file():
        raise ValueError(f"Ожидался файл отчёта: {name}")

    # Проверяем хеш именно тех байтов, которые затем разбираем.
    with path.open("rb") as file:
        content = file.read(max_report_bytes + 1)

    if not content or len(content) > max_report_bytes:
        raise ValueError(f"Недопустимый размер отчёта: {name}")

    actual_hash = hashlib.sha256(content).hexdigest()

    if not hmac.compare_digest(actual_hash, expected_hash):
        raise ValueError(
            f"Изменился отчёт: {name}. "
            "Не обновляй ожидаемый хеш автоматически."
        )

    report = json.loads(
        content.decode("utf-8"),
        parse_constant=reject_json_constant,
        parse_float=finite_json_float,
        object_pairs_hook=unique_json_keys,
    )

    if not isinstance(report, dict):
        raise ValueError(f"Ожидался JSON-объект: {name}")

    return report


def load_experiment_reports():
    # Результат публикуется только после загрузки всех отчётов.
    return {
        name: read_verified_report(name)
        for name in reports_manifest
    }


experiment_reports = None
experiment_reports = load_experiment_reports()

print("Проверено и загружено отчётов:", len(experiment_reports))
print("SHA-256 совпадают.")
print("Размеры ограничены, повторные ключи и NaN/Inf запрещены.")
print("Веса моделей не загружались, эксперименты не перезапускались.")

## E01–E02. Готовая ECAPA и треск
Гипотеза: готовая модель различает голоса на коротких записях, но импульсные помехи могут ухудшать сравнение. Проверили clean-пары и несколько реализаций треска. Ниже читаем сохранённый результат; инференс и генерацию помех повторно не запускаем.

In [ ]:
def load_demo_results():
    path = (
        project_dir / "results/demo_baseline.csv"
    ).resolve(strict=True)

    if not path.is_relative_to(project_dir) or not path.is_file():
        raise ValueError("Недопустимый путь демонстрационного отчёта")

    with path.open("rb") as file:
        content = file.read(16 * 1024 + 1)

    if not content or len(content) > 16 * 1024:
        raise ValueError("Недопустимый размер демонстрационного отчёта")

    expected_hash = (
        "07e6cb8b528b1cf9564cbc5d08b84f509f42afe9be663a2d8af3ef87a4ec5130"
    )

    if not hmac.compare_digest(
        hashlib.sha256(content).hexdigest(),
        expected_hash,
    ):
        raise ValueError("Изменился демонстрационный отчёт")

    reader = csv.DictReader(io.StringIO(content.decode("utf-8")))

    expected_columns = {
        "noise",
        "same_min",
        "different_max",
        "false_different",
        "same_trials",
        "false_same",
        "different_trials",
    }

    if (
        reader.fieldnames is None
        or len(reader.fieldnames) != len(expected_columns)
        or set(reader.fieldnames) != expected_columns
    ):
        raise ValueError("Неожиданные столбцы CSV")

    results = []

    for row in reader:
        same_min = float(row["same_min"])
        different_max = float(row["different_max"])
        false_different = int(row["false_different"])
        same_trials = int(row["same_trials"])
        false_same = int(row["false_same"])
        different_trials = int(row["different_trials"])

        if not all(
            math.isfinite(value) and -1 <= value <= 1
            for value in (same_min, different_max)
        ):
            raise ValueError("Некорректные score в отчёте")

        if not (
            same_trials > 0
            and different_trials > 0
            and 0 <= false_different <= same_trials
            and 0 <= false_same <= different_trials
        ):
            raise ValueError("Некорректные счётчики ошибок")

        results.append({
            "noise": row["noise"],
            "same_min": same_min,
            "different_max": different_max,
            "false_same": false_same,
            "different_trials": different_trials,
            "false_different": false_different,
            "same_trials": same_trials,
        })

    if [row["noise"] for row in results] != [
        "clean", "weak", "medium", "strong"
    ]:
        raise ValueError("Неожиданные уровни помех")

    return results


demo_results = None
demo_results = load_demo_results()

print(
    f"{'Условие':<10} {'SAME min':>10} {'DIFF max':>10}"
    f"{'Ложные SAME':>16} {'Ложные DIFF':>16}"
)

for row in demo_results:
    false_same_text = (
        f"{row['false_same']}/{row['different_trials']}"
    )
    false_different_text = (
        f"{row['false_different']}/{row['same_trials']}"
    )

    print(
        f"{row['noise']:<10}"
        f"{row['same_min']:>10.4f}"
        f"{row['different_max']:>10.4f}"
        f"{false_same_text:>16}"
        f"{false_different_text:>16}"
    )

print("\nПоказан сохранённый результат, без повторного инференса.")

Результат: без помех демонстрационные пары разделились правильно. При сильном треске появились 3 ложных совпадения из 40 сравнений разных дикторов.
Решение: ECAPA оставили основой, но перешли к оценке на большем числе дикторов и раздельной проверке расположения помех. Повторения треска на тех же записях не являются независимыми примерами новых людей. CSV содержит округлённые сводки без полного списка score и рабочего порога, поэтому восстановить по нему ROC или заново подобрать порог нельзя.

## E03. Baseline на LibriSpeech
Гипотеза: правильные ответы на трёх демонстрационных записях недостаточны для оценки качества. Подготовили 120 записей от 40 дикторов, разделённых на dev/test по людям; длительность 3–5 секунд, частота 8 кГц. Порог выбирали только на dev и фиксировали перед test.

In [ ]:
def prepare_librispeech_protocol():
    dataset_dir = (
        project_dir / "data/librispeech_8k"
    ).resolve(strict=True)

    if not dataset_dir.is_relative_to(project_dir):
        raise ValueError("Каталог данных находится вне проекта")

    manifest_path = (
        dataset_dir / "manifest.csv"
    ).resolve(strict=True)

    if not manifest_path.is_relative_to(dataset_dir):
        raise ValueError("Manifest находится вне каталога данных")

    with manifest_path.open("rb") as file:
        content = file.read(128 * 1024 + 1)

    if not content or len(content) > 128 * 1024:
        raise ValueError("Недопустимый размер manifest")

    expected_hash = (
        "daf932b07cfdae91b5491d396c3026b90eb2b8a3c7d5f972554b69df1bdf4e6e"
    )

    if not hmac.compare_digest(
        hashlib.sha256(content).hexdigest(),
        expected_hash,
    ):
        raise ValueError("Изменился LibriSpeech manifest")

    reader = csv.DictReader(io.StringIO(content.decode("utf-8")))

    expected_columns = {
        "split",
        "speaker_id",
        "utterance_id",
        "source_path",
        "path",
        "duration",
        "sample_rate",
    }

    if (
        reader.fieldnames is None
        or len(reader.fieldnames) != len(expected_columns)
        or set(reader.fieldnames) != expected_columns
    ):
        raise ValueError("Неожиданные столбцы manifest")

    records = list(reader)

    if len(records) != 120:
        raise ValueError("Ожидалось 120 записей")

    seen_ids = set()
    seen_paths = set()
    by_split = {"dev": [], "test": []}

    for row in records:
        split = row["split"]

        if split not in by_split:
            raise ValueError("Неожиданное разбиение")

        duration = float(row["duration"])

        if (
            not math.isfinite(duration)
            or not 3 <= duration <= 5
            or int(row["sample_rate"]) != 8000
        ):
            raise ValueError("Некорректная длительность или частота")

        if not row["speaker_id"] or not row["utterance_id"]:
            raise ValueError("Отсутствует идентификатор")

        audio_path = Path(row["path"]).resolve(strict=True)

        if (
            not audio_path.is_relative_to(dataset_dir / split)
            or not audio_path.is_file()
        ):
            raise ValueError("Недопустимый путь аудиозаписи")

        if (
            row["utterance_id"] in seen_ids
            or audio_path in seen_paths
        ):
            raise ValueError("Обнаружена повторная запись")

        seen_ids.add(row["utterance_id"])
        seen_paths.add(audio_path)
        by_split[split].append(row)

    speakers = {
        split: {row["speaker_id"] for row in rows}
        for split, rows in by_split.items()
    }

    if speakers["dev"] & speakers["test"]:
        raise ValueError("Есть пересечение dev/test по дикторам")

    protocol = {}

    for split, rows in by_split.items():
        if len(rows) != 60 or len(speakers[split]) != 20:
            raise ValueError("Неожиданный размер разбиения")

        if any(
            sum(row["speaker_id"] == speaker for row in rows) != 3
            for speaker in speakers[split]
        ):
            raise ValueError("Ожидалось по три записи на диктора")

        pairs = [
            {
                "first_id": first["utterance_id"],
                "second_id": second["utterance_id"],
                "is_same": first["speaker_id"] == second["speaker_id"],
            }
            for first, second in combinations(rows, 2)
        ]

        same_count = sum(pair["is_same"] for pair in pairs)

        if same_count != 60 or len(pairs) - same_count != 1710:
            raise ValueError("Неожиданное количество пар")

        protocol[split] = {
            "records": rows,
            "pairs": pairs,
            "speaker_count": len(speakers[split]),
        }

    return protocol


librispeech_protocol = None
librispeech_protocol = prepare_librispeech_protocol()

for split, data in librispeech_protocol.items():
    same_count = sum(pair["is_same"] for pair in data["pairs"])

    print(
        f"{split}: "
        f"{len(data['records'])} записей, "
        f"{data['speaker_count']} дикторов, "
        f"SAME {same_count}, "
        f"DIFFERENT {len(data['pairs']) - same_count}"
    )

print("\nПересечений дикторов dev/test нет.")
print("Протокол восстановлен; аудиоинференс не выполнялся.")
print("Хеш проверен для manifest, а не для содержимого всех WAV.")

In [ ]:
historical_baseline = {
    "dev": {
        "false_same": 44,
        "different_count": 1710,
        "false_different": 0,
        "same_count": 60,
    },
    "test": {
        "false_same": 28,
        "different_count": 1710,
        "false_different": 2,
        "same_count": 60,
    },
}

print("ИСТОРИЧЕСКИЙ РЕЗУЛЬТАТ — без повторного инференса")
print("Порог первоначального запуска: 0.402372\n")

print(
    f"{'Разбиение':<12}"
    f"{'FAR':>10}"
    f"{'FRR':>10}"
    f"{'Ошибка':>10}"
)

for split, counts in historical_baseline.items():
    far = counts["false_same"] / counts["different_count"]
    frr = counts["false_different"] / counts["same_count"]

    print(
        f"{split:<12}"
        f"{far:>10.2%}"
        f"{frr:>10.2%}"
        f"{(far + frr) / 2:>10.2%}"
    )

print("\nИсточник: сохранённый вывод исследовательского notebook.")
print("Счётчики перенесены явно; метрики рассчитаны из них.")

Решение: приняли ECAPA как baseline. Разделение по дикторам исключает присутствие одного человека одновременно в dev и test. Далее проверяем адаптацию к треску; результаты этого clean-протокола не смешиваем с последующими шумовыми сценариями.

## E04. Адаптация на синтетическом треске
Гипотеза: обучение последнего слоя ECAPA улучшит устойчивость к помехам без полного переобучения encoder. Использовали LibriSpeech train-clean-100: 251 диктор, фрагменты по 3 секунды, classification loss и синтетический треск. Обучали fc и новую классификационную голову в течение 300 шагов; выбор checkpoint, калибровку порога и test разделили по дикторам.

In [ ]:
def summarize_synthetic_adaptation():
    report = read_verified_report("synthetic_adaptation")

    if (
        report["model"] != "speechbrain/spkrec-ecapa-voxceleb"
        or report["adaptation"] != "fc_only"
        or report["best_step"] != 300
        or report["decision_rule"] != "similarity > threshold"
    ):
        raise ValueError("Неожиданная конфигурация эксперимента")

    groups = {
        "checkpoint": report["checkpoint_speakers"],
        "calibration": report["calibration_speakers"],
        "test": report["test_speakers"],
    }

    expected_counts = {
        "checkpoint": 10,
        "calibration": 10,
        "test": 20,
    }

    for name, speakers in groups.items():
        if (
            not isinstance(speakers, list)
            or not all(isinstance(speaker, str) for speaker in speakers)
            or len(speakers) != expected_counts[name]
            or len(set(speakers)) != len(speakers)
        ):
            raise ValueError(f"Некорректная группа дикторов: {name}")

    for first, second in combinations(groups, 2):
        if set(groups[first]) & set(groups[second]):
            raise ValueError("Пересечение групп дикторов")

    expected_scenarios = (
        "clean",
        "weak/first",
        "weak/second",
        "weak/both",
        "medium/first",
        "medium/second",
        "medium/both",
        "strong/first",
        "strong/second",
        "strong/both",
    )

    for name in ("baseline", "adapted"):
        evaluation = report[f"{name}_test"]
        threshold = evaluation["threshold"]

        if (
            isinstance(threshold, bool)
            or not isinstance(threshold, (int, float))
            or not math.isfinite(threshold)
            or not -1 <= threshold <= 1
        ):
            raise ValueError(f"Некорректный порог: {name}")

        if not math.isclose(
            threshold,
            report[f"{name}_threshold"],
            rel_tol=0,
            abs_tol=1e-12,
        ):
            raise ValueError(f"Не совпали пороги: {name}")

        scenarios = evaluation["scenarios"]

        if set(scenarios) != set(expected_scenarios):
            raise ValueError(f"Неожиданные сценарии: {name}")

        errors = []

        for scenario in expected_scenarios:
            metrics = scenarios[scenario]

            for key in ("far", "frr", "balanced_error"):
                value = metrics[key]

                if (
                    isinstance(value, bool)
                    or not isinstance(value, (int, float))
                    or not math.isfinite(value)
                    or not 0 <= value <= 1
                ):
                    raise ValueError(
                        f"Некорректная метрика: {name}/{scenario}/{key}"
                    )

            calculated_error = (
                metrics["far"] + metrics["frr"]
            ) / 2

            if not math.isclose(
                calculated_error,
                metrics["balanced_error"],
                rel_tol=0,
                abs_tol=1e-12,
            ):
                raise ValueError("Не совпала ошибка сценария")

            errors.append(calculated_error)

        recorded_mean = evaluation["balanced_error"]

        if (
            isinstance(recorded_mean, bool)
            or not isinstance(recorded_mean, (int, float))
            or not math.isfinite(recorded_mean)
            or not math.isclose(
                float(np.mean(errors)),
                recorded_mean,
                rel_tol=0,
                abs_tol=1e-12,
            )
        ):
            raise ValueError("Не совпала средняя ошибка")

    return report, expected_scenarios


synthetic_adaptation_summary = None
synthetic_adaptation_summary = summarize_synthetic_adaptation()

report, scenario_names = synthetic_adaptation_summary

print("СОХРАНЁННЫЙ РЕЗУЛЬТАТ — без повторного обучения и инференса")
print(f"Порог baseline: {report['baseline_threshold']:.6f}")
print(f"Порог adapted:  {report['adapted_threshold']:.6f}\n")

print(
    f"{'Сценарий':<16}"
    f"{'FAR base':>11}"
    f"{'FRR base':>11}"
    f"{'FAR adapt':>11}"
    f"{'FRR adapt':>11}"
)

for scenario in scenario_names:
    baseline = report["baseline_test"]["scenarios"][scenario]
    adapted = report["adapted_test"]["scenarios"][scenario]

    print(
        f"{scenario:<16}"
        f"{baseline['far']:>11.2%}"
        f"{baseline['frr']:>11.2%}"
        f"{adapted['far']:>11.2%}"
        f"{adapted['frr']:>11.2%}"
    )

baseline_error = report["baseline_test"]["balanced_error"]
adapted_error = report["adapted_test"]["balanced_error"]

print(f"\nСредняя ошибка baseline: {baseline_error:.2%}")
print(f"Средняя ошибка adapted:  {adapted_error:.2%}")
print(
    "Изменение adapted − baseline: "
    f"{100 * (adapted_error - baseline_error):+.2f} п.п."
)
print("Проверены разбиения и согласованность метрик отчёта.")

Результат: средняя ошибка по десяти сценариям снизилась с 5.21% до 4.31%. Улучшение неоднородно: например, при сильном треске только в первой записи FRR вырос с 18.33% до 20.00%.
Решение: сохранили адаптацию как полезный эксперимент, но не объявляли универсальным улучшением. Далее проверили перенос на естественный шум Vibravox. Пороги у моделей собственные, поэтому сравниваем системы «модель + настроенный порог», а не только изменение весов.
Ограничение: проверка хеша и формул подтверждает целостность и внутреннюю согласованность отчёта, но не воспроизводит обучение и не доказывает достоверность исходных измерений.

## E05–E06. Перенос на Vibravox и выбор порога
Гипотеза: порог, выбранный на LibriSpeech, может плохо переноситься на естественный шум Vibravox. Сначала проверили обе модели на 120 test-записях с прежними порогами, затем выбрали новые пороги на 102 validation-записях других дикторов. На test пороги не подбирали.

In [ ]:
# Проценты из сохранённых выводов исследовательского notebook.
# Это округлённые исторические результаты, не новый инференс.

domain_transfer_history = (
    # Сценарий, модель, источник порога, FAR %, FRR %
    ("clean-clean", "baseline", "LibriSpeech", 19.47, 0.00),
    ("clean-clean", "baseline", "Vibravox",    2.92, 0.00),
    ("clean-clean", "adapted",  "LibriSpeech", 15.32, 0.00),
    ("clean-clean", "adapted",  "Vibravox",    2.40, 0.00),

    ("clean-noisy", "baseline", "LibriSpeech", 15.20, 0.00),
    ("clean-noisy", "baseline", "Vibravox",    1.17, 11.67),
    ("clean-noisy", "adapted",  "LibriSpeech", 12.46, 0.00),
    ("clean-noisy", "adapted",  "Vibravox",    1.26, 11.67),

    ("noisy-noisy", "baseline", "LibriSpeech", 24.68, 0.00),
    ("noisy-noisy", "baseline", "Vibravox",    3.27, 0.00),
    ("noisy-noisy", "adapted",  "LibriSpeech", 22.46, 0.00),
    ("noisy-noisy", "adapted",  "Vibravox",    3.80, 0.00),
)


def summarize_domain_transfer(rows):
    scenarios = {"clean-clean", "clean-noisy", "noisy-noisy"}
    models = {"baseline", "adapted"}
    threshold_sources = {"LibriSpeech", "Vibravox"}

    expected = {
        (scenario, model, source)
        for scenario in scenarios
        for model in models
        for source in threshold_sources
    }

    seen = set()
    errors = {}

    for scenario, model, source, far_percent, frr_percent in rows:
        key = (scenario, model, source)

        if key not in expected or key in seen:
            raise ValueError("Неожиданная или повторная строка")

        if not all(
            isinstance(value, (int, float))
            and not isinstance(value, bool)
            and math.isfinite(value)
            and 0 <= value <= 100
            for value in (far_percent, frr_percent)
        ):
            raise ValueError("Некорректные проценты ошибок")

        seen.add(key)
        errors[key] = (far_percent + frr_percent) / 2

    if seen != expected:
        raise ValueError("Не хватает результатов сравнения")

    return errors


domain_transfer_errors = None
domain_transfer_errors = summarize_domain_transfer(
    domain_transfer_history
)

print("ИСТОРИЧЕСКИЙ TEST — без повторного инференса\n")
print(
    f"{'Сценарий':<15}"
    f"{'Модель':<10}"
    f"{'Порог из':<13}"
    f"{'FAR':>9}"
    f"{'FRR':>9}"
    f"{'Ошибка':>10}"
)

for scenario, model, source, far, frr in domain_transfer_history:
    error = domain_transfer_errors[scenario, model, source]

    print(
        f"{scenario:<15}"
        f"{model:<10}"
        f"{source:<13}"
        f"{far:>8.2f}%"
        f"{frr:>8.2f}%"
        f"{error:>9.2f}%"
    )

print("\nСредняя ошибка — равный вес трёх сценариев:")

for model in ("baseline", "adapted"):
    means = {}

    for source in ("LibriSpeech", "Vibravox"):
        means[source] = float(np.mean([
            domain_transfer_errors[scenario, model, source]
            for scenario in ("clean-clean", "clean-noisy", "noisy-noisy")
        ]))

    print(
        f"{model:<10}: "
        f"{means['LibriSpeech']:.2f}% → "
        f"{means['Vibravox']:.2f}%"
    )

print("\nМетрики пересчитаны из округлённых исторических процентов.")
print("Новые пороги здесь не выбирались.")

Как выбирали новый порог: перебирали значения validation-score, считали FAR и FRR отдельно для clean-clean, clean-noisy и noisy-noisy, минимизировали среднюю сбалансированную ошибку. При равенстве предпочитали меньший средний FAR.
Получили validation-пороги около 0.515286 для baseline и 0.495882 для ранней adapted. Это исторические пороги данного эксперимента; их не переносим в итоговый интерфейс.

Результат: после настройки порога на целевом домене средняя test-ошибка снизилась с 9.89% до 3.17% у baseline и с 8.37% до 3.19% у adapted. Одновременно выросли пропуски clean-noisy — до 11.67%.
Решение: подтвердили необходимость настройки порога на целевых данных. Преимущество ранней адаптации после этого исчезло по общей метрике. Следующий шаг — разобрать оставшиеся ошибочные пары, а не сразу усложнять модель.
Ограничение: таблица перенесена из сохранённого вывода; это документирование результата, а не повторная оценка. Первоначальный test впоследствии многократно анализировался, поэтому его нельзя считать полностью нетронутой финальной проверкой.

## E07. Анализ ошибочных пар

**Гипотеза:** пропуски сосредоточены у отдельных дикторов или записей, а не распределены равномерно.

Для каждой модели выделили SAME-пары, не прошедшие её порог, сравнили списки ошибок и сгруппировали их по дикторам. Затем прослушали проблемные и удачные записи.

In [ ]:

def analyze_same_pair_errors(pairs, threshold):
    if (
        isinstance(threshold, bool)
        or not isinstance(threshold, (int, float))
        or not math.isfinite(threshold)
        or not -1 <= threshold <= 1
    ):
        raise ValueError("Некорректный порог")

    missed_pairs = set()
    seen_pairs = set()
    same_count = 0

    for pair in pairs:
        first_id = pair["first_id"]
        second_id = pair["second_id"]
        score = pair["similarity"]
        is_same = pair["is_same"]

        if (
            not isinstance(first_id, str)
            or not isinstance(second_id, str)
            or not first_id
            or not second_id
            or first_id == second_id
        ):
            raise ValueError("Некорректные идентификаторы пары")

        if (
            type(is_same) is not bool
            or isinstance(score, bool)
            or not isinstance(score, (int, float))
            or not math.isfinite(score)
            or not -1.00001 <= score <= 1.00001
        ):
            raise ValueError("Некорректная метка или similarity")

        key = tuple(sorted((first_id, second_id)))

        if key in seen_pairs:
            raise ValueError("Повторная пара")

        seen_pairs.add(key)

        if is_same:
            same_count += 1

            if score <= threshold:
                missed_pairs.add(key)

    if same_count == 0:
        raise ValueError("В выборке отсутствуют SAME-пары")

    return {
        "same_count": same_count,
        "missed_pairs": missed_pairs,
        "frr": len(missed_pairs) / same_count,
    }


print("Функция анализа сохранена.")
print("Для сравнения моделей используются пересечение и разность")
print("наборов missed_pairs. На этом шаге анализ заново не запускался.")

In [ ]:
historical_misses_by_speaker = {
    "yP5TS6EloW": 8,
    "5eh1ULfUPE": 4,
    "6ft7Ezz5sQ": 4,
    "3OxxtNhkuv": 2,
    "IrXN65uF2D": 2,
    "ll30-whTFY": 1,
}

total_speakers = 20
same_pairs_per_speaker = 9
total_same_pairs = total_speakers * same_pairs_per_speaker

if not all(
    type(count) is int
    and 0 < count <= same_pairs_per_speaker
    for count in historical_misses_by_speaker.values()
):
    raise ValueError("Некорректные счётчики пропусков")

missed_count = sum(historical_misses_by_speaker.values())

if missed_count != 21:
    raise ValueError("Не совпало историческое число пропусков")

print("ИСТОРИЧЕСКИЙ АНАЛИЗ — без повторного инференса\n")
print(f"{'Диктор':<16}{'Пропуски / SAME-пары':>23}")

for speaker, count in sorted(
    historical_misses_by_speaker.items(),
    key=lambda item: (-item[1], item[0]),
):
    print(f"{speaker:<16}{count:>15}/{same_pairs_per_speaker}")

print(
    f"\nВсего пропусков: {missed_count}/{total_same_pairs} "
    f"({missed_count / total_same_pairs:.2%})"
)
print(
    "Дикторов с пропусками: "
    f"{len(historical_misses_by_speaker)}/{total_speakers}"
)

print("\nПо сохранённому первоначальному анализу:")
print("Baseline и ранняя adapted пропустили одни и те же 21 пару.")
print("Ошибок только одной из моделей не было.")
print("Идентичность списков здесь повторно не проверяется:")
print("для этого нужны исходные поэлементные score и идентификаторы пар.")

Результат: 21 пропуск из 180 SAME-пар сосредоточился у шести дикторов. У обеих моделей ошибочные пары совпали. Прослушивание показало различия фона и произношения; некоторые clean-записи были очень тихими.
Решение: отдельно проверить шумоподавление, удаление пауз, точность PCM16 и нормализацию громкости. Низкий RMS и субъективное звучание считаем гипотезами, а не доказанными причинами ошибок. RMS шумной записи не является оценкой громкости речи или SNR

## E08. Шумоподавление

**Гипотеза:** удаление фонового шума улучшит сравнение голосов.

Проверили нестационарное спектральное шумоподавление с интенсивностями 0.5, 0.8 и 1.0 на 102 validation-записях. Сначала использовали исходные пороги, затем отдельно настроили порог для каждого варианта обработки. Усиление и удаление пауз не применялись.

In [ ]:
def denoise_for_experiment(signal, sample_rate, strength=0.5):
    # Необязательная зависимость нужна только при вызове функции.
    import noisereduce as nr

    if sample_rate != 8000:
        raise ValueError("Эксперимент рассчитан на частоту 8000 Гц")

    if (
        isinstance(strength, bool)
        or strength not in (0.5, 0.8, 1.0)
    ):
        raise ValueError("Допустимая интенсивность: 0.5, 0.8 или 1.0")

    signal = np.asarray(signal)

    if (
        signal.ndim != 1
        or signal.dtype.kind != "f"
        or not 8000 <= signal.size <= 40000
        or not np.isfinite(signal).all()
    ):
        raise ValueError("Нужен конечный mono-сигнал длиной 1–5 секунд")

    processed = nr.reduce_noise(
        y=signal.copy(),
        sr=sample_rate,
        stationary=False,
        prop_decrease=strength,
        n_fft=512,
        win_length=256,
        hop_length=64,
        time_constant_s=0.5,
        n_jobs=1,
        use_torch=False,
    )

    processed = np.asarray(processed, dtype=np.float32)

    if (
        processed.shape != signal.shape
        or not np.isfinite(processed).all()
    ):
        raise RuntimeError("Некорректный результат шумоподавления")

    return processed


print("Функция обработки сохранена.")
print("Установка пакетов, обработка аудио и инференс не выполнялись.")

In [ ]:


denoise_history = (
    ("original",    "baseline",  2.68, 0.515286, 2.68),
    ("original",    "adapted",   2.49, 0.495882, 2.49),

    ("denoise_0.5", "baseline",  3.11, 0.495624, 2.68),
    ("denoise_0.5", "adapted",   2.96, 0.476600, 2.46),

    ("denoise_0.8", "baseline",  3.24, 0.523600, 3.12),
    ("denoise_0.8", "adapted",   3.12, 0.483896, 2.89),

    ("denoise_1.0", "baseline", 10.18, 0.573144, 8.70),
    ("denoise_1.0", "adapted",  10.41, 0.563576, 8.78),
)


def validate_denoise_history(rows):
    expected = {
        (treatment, model)
        for treatment in (
            "original", "denoise_0.5", "denoise_0.8", "denoise_1.0"
        )
        for model in ("baseline", "adapted")
    }

    seen = set()

    for treatment, model, old_error, threshold, new_error in rows:
        key = (treatment, model)

        if key not in expected or key in seen:
            raise ValueError("Неожиданная или повторная строка")

        if not all(
            isinstance(value, (int, float))
            and not isinstance(value, bool)
            and math.isfinite(value)
            for value in (old_error, threshold, new_error)
        ):
            raise ValueError("Некорректные числовые значения")

        if not (
            0 <= old_error <= 100
            and 0 <= new_error <= 100
            and -1 <= threshold <= 1
        ):
            raise ValueError("Значения находятся вне допустимого диапазона")

        seen.add(key)

    if seen != expected:
        raise ValueError("Не хватает результатов")

    return {
        model: new_error
        for treatment, model, _, _, new_error in rows
        if treatment == "original"
    }


denoise_reference = None
denoise_reference = validate_denoise_history(denoise_history)

print("ИСТОРИЧЕСКИЙ VALIDATION — без повторной обработки\n")
print(
    f"{'Обработка':<14}"
    f"{'Модель':<10}"
    f"{'Старый порог':>14}"
    f"{'Новый порог':>14}"
    f"{'После':>9}"
    f"{'Δ, п.п.':>10}"
)

for treatment, model, old_error, threshold, new_error in denoise_history:
    delta = new_error - denoise_reference[model]

    print(
        f"{treatment:<14}"
        f"{model:<10}"
        f"{old_error:>13.2f}%"
        f"{threshold:>14.6f}"
        f"{new_error:>8.2f}%"
        f"{delta:>+10.2f}"
    )

print("\nСтолбцы «Старый порог» и «После» — средняя ошибка.")
print("Δ сравнивает обработку с original той же модели.")
print("Источник: сохранённые выводы исследовательского notebook.")
print("Пороги здесь не подбирались; test не использовался.")

**Результат:** после отдельной настройки порога умеренная очистка 0.5 дала 2.46% против 2.49% у ранней adapted. Интенсивности 0.8 и 1.0 ухудшили результат; сильная очистка существенно ухудшила обе модели.

**Решение:** шумоподавление не включили в основной конвейер. Выигрыш 0.03 п.п. на небольшой validation-выборке не обосновывает дополнительную обработку без независимого подтверждения.

**Ограничение:** улучшение звучания при прослушивании не гарантирует сохранения признаков диктора. Функция выше сохраняет параметры обработки, но сама не воспроизводит полную оценку: для неё нужны исходные записи, обе модели, пары и выбор порогов.

## E09. Удаление пауз

**Гипотеза:** длинные неречевые участки ухудшают представление короткой записи.

На тех же 102 validation-записях удаляли низкоэнергетические промежутки от 400 мс, сохраняя по 120 мс у внутренних границ. Если оставалось меньше секунды или половины записи, сохраняли оригинал. Шумоподавление не применяли, пороги оставили исходными.

In [ ]:
def shorten_pauses_for_experiment(signal, sample_rate=8000):
    import librosa

    signal = np.asarray(signal)

    if (
        sample_rate != 8000
        or signal.ndim != 1
        or signal.dtype.kind != "f"
        or not 8000 <= signal.size <= 40000
        or not np.isfinite(signal).all()
    ):
        raise ValueError("Нужен конечный mono-сигнал 8 кГц, 1–5 секунд")

    intervals = librosa.effects.split(
        signal,
        top_db=35,
        frame_length=256,
        hop_length=80,
    )

    if len(intervals) == 0:
        return signal.copy(), {
            "removed_seconds": 0.0,
            "status": "no_intervals_original_kept",
        }

    min_gap = round(0.4 * sample_rate)
    padding = round(0.12 * sample_rate)
    keep = np.ones(signal.size, dtype=bool)

    previous_end = 0
    gaps = []

    for start, end in intervals:
        start, end = int(start), int(end)

        if not previous_end <= start < end <= signal.size:
            raise RuntimeError("Некорректные интервалы обработки")

        gaps.append((previous_end, start))
        previous_end = end

    gaps.append((previous_end, signal.size))

    for start, end in gaps:
        if end - start < min_gap:
            continue

        cut_start = start + padding if start > 0 else start
        cut_end = end - padding if end < signal.size else end

        if cut_end > cut_start:
            keep[cut_start:cut_end] = False

    if keep.sum() < sample_rate or keep.mean() < 0.5:
        return signal.copy(), {
            "removed_seconds": 0.0,
            "status": "excessive_cut_original_kept",
        }

    processed = signal[keep].copy()

    return processed, {
        "removed_seconds": (signal.size - processed.size) / sample_rate,
        "status": "processed",
    }


print("Логика сокращения пауз сохранена.")
print("Обработка аудио не запускалась.")

In [ ]:
# Сценарий, модель, FAR original, FRR original, FAR pauses, FRR pauses.
# Все показатели указаны в процентах из сохранённого вывода.

pause_history = (
    ("clean-clean", "baseline", 2.61, 0.00, 2.21, 0.00),
    ("clean-noisy", "baseline", 1.06, 7.84, 0.98, 9.80),
    ("noisy-noisy", "baseline", 4.58, 0.00, 4.58, 0.00),

    ("clean-clean", "adapted", 2.12, 0.00, 1.55, 1.96),
    ("clean-noisy", "adapted", 0.98, 7.19, 1.02, 9.15),
    ("noisy-noisy", "adapted", 4.66, 0.00, 4.66, 0.00),
)


def summarize_pause_history(rows):
    expected = {
        (scenario, model)
        for scenario in ("clean-clean", "clean-noisy", "noisy-noisy")
        for model in ("baseline", "adapted")
    }

    seen = set()
    errors = {
        model: {"original": [], "pauses": []}
        for model in ("baseline", "adapted")
    }

    for scenario, model, far, frr, pause_far, pause_frr in rows:
        key = (scenario, model)

        if key not in expected or key in seen:
            raise ValueError("Неожиданная или повторная строка")

        if not all(
            isinstance(value, (int, float))
            and not isinstance(value, bool)
            and math.isfinite(value)
            and 0 <= value <= 100
            for value in (far, frr, pause_far, pause_frr)
        ):
            raise ValueError("Некорректные метрики")

        seen.add(key)
        errors[model]["original"].append((far + frr) / 2)
        errors[model]["pauses"].append((pause_far + pause_frr) / 2)

    if seen != expected:
        raise ValueError("Не хватает результатов")

    return {
        model: {
            treatment: float(np.mean(values))
            for treatment, values in treatments.items()
        }
        for model, treatments in errors.items()
    }


pause_summary = None
pause_summary = summarize_pause_history(pause_history)

print("ИСТОРИЧЕСКИЙ VALIDATION — без повторной обработки\n")
print(
    f"{'Модель':<10}{'Сценарий':<15}"
    f"{'FAR исход.':>12}{'FRR исход.':>12}"
    f"{'FAR паузы':>12}{'FRR паузы':>12}"
)

for scenario, model, far, frr, pause_far, pause_frr in pause_history:
    print(
        f"{model:<10}{scenario:<15}"
        f"{far:>11.2f}%{frr:>11.2f}%"
        f"{pause_far:>11.2f}%{pause_frr:>11.2f}%"
    )

print("\nСредняя ошибка — равный вес трёх сценариев:")

for model, result in pause_summary.items():
    print(
        f"{model:<10}: "
        f"{result['original']:.2f}% → {result['pauses']:.2f}%"
    )

print("\nПроценты перенесены из сохранённого вывода.")
print("Новые пороги не выбирались; test не использовался.")

**Результат:** средняя ошибка baseline выросла с 2.68% до 2.93%, ранней adapted — с 2.49% до 3.06%. Особенно ухудшились пропуски clean-noisy.

**Решение:** этот способ удаления пауз не включили в основной конвейер.

**Ограничение:** проверяли конкретную энергетическую эвристику при фиксированных порогах. Это не доказывает бесполезность любого VAD. Низкая энергия не обязательно означает отсутствие речи; повторная настройка порога для этого варианта здесь не выполнялась.

## E10. PCM16 против float32

**Гипотеза:** сохранение тихих записей в PCM16 могло ухудшить голосовые признаки.

Повторно прочитали исходные байты тех же 102 validation-записей, выполнили ресемплинг и сохранили float32-кеш без промежуточного PCM16. Сравнили погрешность сигналов и результаты верификации при прежних порогах.

In [ ]:
def measure_pcm_error(reference, saved):
    reference = np.asarray(reference)
    saved = np.asarray(saved)

    if (
        reference.ndim != 1
        or reference.shape != saved.shape
        or reference.dtype != np.dtype("float32")
        or saved.dtype != np.dtype("float32")
        or not 8000 <= reference.size <= 40000
        or not np.isfinite(reference).all()
        or not np.isfinite(saved).all()
    ):
        raise ValueError(
            "Нужны два конечных float32 mono-сигнала "
            "одинаковой длины, 1–5 секунд при 8 кГц"
        )

    reference64 = reference.astype(np.float64)
    saved64 = saved.astype(np.float64)
    error = saved64 - reference64

    signal_power = float(np.mean(reference64 ** 2))
    error_power = float(np.mean(error ** 2))

    return {
        "rms_dbfs": float(
            10 * np.log10(max(signal_power, 1e-24))
        ),
        "error_snr_db": float(
            10 * np.log10(
                max(signal_power, 1e-24)
                / max(error_power, 1e-24)
            )
        ),
        "max_abs_error": float(np.max(np.abs(error))),
        "outside_pcm_range_fraction": float(np.mean(
            (reference64 < -1.0) | (reference64 >= 1.0)
        )),
    }


print("Функция диагностики сохранена.")
print("error_snr_db — отношение сигнала к ошибке сохранения,")
print("а не отношение речи к акустическому шуму.")
print("Загрузка исходных аудиобайтов и сравнение не запускались.")

In [ ]:
# Сценарий, модель, FAR PCM16, FRR PCM16, FAR float32, FRR float32.
# Значения в процентах из сохранённого вывода.

pcm_history = (
    ("clean-clean", "baseline", 2.61, 0.00, 2.78, 0.00),
    ("clean-noisy", "baseline", 1.06, 7.84, 0.98, 7.84),
    ("noisy-noisy", "baseline", 4.58, 0.00, 4.66, 0.00),

    ("clean-clean", "adapted", 2.12, 0.00, 2.21, 0.00),
    ("clean-noisy", "adapted", 0.98, 7.19, 0.98, 7.19),
    ("noisy-noisy", "adapted", 4.66, 0.00, 4.82, 0.00),
)


def summarize_preprocessing_comparison(rows):
    expected = {
        (scenario, model)
        for scenario in ("clean-clean", "clean-noisy", "noisy-noisy")
        for model in ("baseline", "adapted")
    }

    seen = set()
    errors = {
        model: {"before": [], "after": []}
        for model in ("baseline", "adapted")
    }

    for scenario, model, far_before, frr_before, far_after, frr_after in rows:
        key = (scenario, model)

        if key not in expected or key in seen:
            raise ValueError("Неожиданная или повторная строка")

        if not all(
            isinstance(value, (int, float))
            and not isinstance(value, bool)
            and math.isfinite(value)
            and 0 <= value <= 100
            for value in (
                far_before, frr_before, far_after, frr_after
            )
        ):
            raise ValueError("Некорректные метрики")

        seen.add(key)
        errors[model]["before"].append(
            (far_before + frr_before) / 2
        )
        errors[model]["after"].append(
            (far_after + frr_after) / 2
        )

    if seen != expected:
        raise ValueError("Не хватает результатов")

    return {
        model: {
            stage: float(np.mean(values))
            for stage, values in stages.items()
        }
        for model, stages in errors.items()
    }


pcm_summary = None
pcm_summary = summarize_preprocessing_comparison(pcm_history)

print("ИСТОРИЧЕСКИЙ VALIDATION — без повторного инференса\n")
print(
    f"{'Модель':<10}{'Сценарий':<15}"
    f"{'FAR PCM16':>12}{'FRR PCM16':>12}"
    f"{'FAR float':>12}{'FRR float':>12}"
)

for scenario, model, far, frr, float_far, float_frr in pcm_history:
    print(
        f"{model:<10}{scenario:<15}"
        f"{far:>11.2f}%{frr:>11.2f}%"
        f"{float_far:>11.2f}%{float_frr:>11.2f}%"
    )

print("\nСредняя ошибка:")

for model, result in pcm_summary.items():
    print(
        f"{model:<10}: "
        f"{result['before']:.2f}% → {result['after']:.2f}%"
    )

print("\nИсторическая диагностика ошибки сохранения:")
print("clean: SNR ошибки min/median/max — 44.80 / 59.59 / 69.29 дБ")
print("noisy: SNR ошибки min/median/max — 57.89 / 65.01 / 72.27 дБ")
print("Максимальная абсолютная ошибка: около 0.00003052")
print("Записей с исходными отсчётами вне диапазона PCM: 0")

print("\nИсточник: сохранённые выводы исследовательского notebook.")
print("Метрики пересчитаны из округлённых процентов.")
print("Пороги не менялись; test не использовался.")

**Результат:** переход на исходный float32 не улучшил качество. Средняя ошибка baseline изменилась с 2.68% до 2.71%, ранней adapted — с 2.49% до 2.53%. Изменения similarity были небольшими.

**Решение:** гипотеза, что PCM16 является основной причиной пропусков на этой выборке, не подтвердилась. Не стали усложнять основной конвейер ради этого изменения.

**Ограничение:** вывод относится к проверенным записям и способу сохранения. Усиление уже сохранённого PCM16 не восстановит потерянные детали; следующим экспериментом отдельно проверили нормализацию исходного float32.

## E11. Нормализация громкости

**Гипотеза:** усиление тихой речи перед инференсом улучшит эмбеддинги.

На 102 validation-записях использовали целевой RMS −25 dBFS, максимальное усиление +20 dB и ограничение пика 0.95. Обрабатывали исходный float32, без очистки и удаления пауз. Пороги не меняли.

In [ ]:
def normalize_volume_for_experiment(signal, sample_rate=8000):
    signal = np.asarray(signal)

    if (
        sample_rate != 8000
        or signal.ndim != 1
        or signal.dtype != np.dtype("float32")
        or not 8000 <= signal.size <= 40000
        or not np.isfinite(signal).all()
    ):
        raise ValueError(
            "Нужен конечный float32 mono-сигнал 8 кГц, 1–5 секунд"
        )

    target_dbfs = -25.0
    max_gain_db = 20.0
    peak_limit = 0.95

    rms = float(np.sqrt(np.mean(
        signal.astype(np.float64) ** 2
    )))
    peak = float(np.max(np.abs(signal)))

    if rms < 1e-12:
        raise ValueError("Сигнал слишком слабый для нормализации")

    gain = min(
        10 ** (target_dbfs / 20) / rms,
        10 ** (max_gain_db / 20),
        peak_limit / max(peak, 1e-12),
    )

    if not math.isfinite(gain) or gain <= 0:
        raise RuntimeError("Некорректный коэффициент усиления")

    processed = (signal * gain).astype(np.float32)

    if (
        processed.shape != signal.shape
        or not np.isfinite(processed).all()
        or np.max(np.abs(processed)) > peak_limit + 1e-6
    ):
        raise RuntimeError("Некорректный результат нормализации")

    return processed, float(20 * np.log10(gain))


print("Функция нормализации сохранена.")
print("Громкий сигнал может быть ослаблен для соблюдения лимита пика.")
print("Обработка аудио и инференс не запускались.")

In [ ]:
# Сценарий, модель, FAR float32, FRR float32, FAR norm, FRR norm.
# Все метрики — проценты из сохранённого вывода.

volume_history = (
    ("clean-clean", "baseline", 2.78, 0.00, 2.78, 0.00),
    ("clean-noisy", "baseline", 0.98, 7.84, 0.98, 7.84),
    ("noisy-noisy", "baseline", 4.66, 0.00, 4.66, 0.00),

    ("clean-clean", "adapted", 2.21, 0.00, 2.21, 0.00),
    ("clean-noisy", "adapted", 0.98, 7.19, 0.94, 7.19),
    ("noisy-noisy", "adapted", 4.82, 0.00, 4.74, 0.00),
)

# Используем функцию проверки и усреднения из E10.
volume_summary = None
volume_summary = summarize_preprocessing_comparison(volume_history)

print("ИСТОРИЧЕСКИЙ VALIDATION — без повторной обработки\n")
print(
    f"{'Модель':<10}{'Сценарий':<15}"
    f"{'FAR float':>12}{'FRR float':>12}"
    f"{'FAR norm':>12}{'FRR norm':>12}"
)

for scenario, model, far, frr, norm_far, norm_frr in volume_history:
    print(
        f"{model:<10}{scenario:<15}"
        f"{far:>11.2f}%{frr:>11.2f}%"
        f"{norm_far:>11.2f}%{norm_frr:>11.2f}%"
    )

print("\nСредняя ошибка: float32 → нормализация float32")

for model, result in volume_summary.items():
    print(
        f"{model:<10}: "
        f"{result['before']:.2f}% → {result['after']:.2f}%"
    )

print("\nИсторическое изменение громкости, min / median / max:")
print("clean:  +0.80 / +10.49 / +20.00 dB")
print("noisy:  −2.17 /  +5.07 / +12.18 dB")
print("Медианное |Δ similarity|: около 0.000025")
print("Максимальное |Δ similarity|: около 0.000170")

print("\nИсточник: сохранённый вывод исследовательского notebook.")
print("Средние рассчитаны из округлённых процентов.")
print("Пороги не менялись; test не использовался.")

**Результат:** средняя ошибка baseline осталась 2.71%, ранней adapted изменилась с 2.53% до 2.51%. Несмотря на заметное изменение громкости, similarity почти не изменился.

**Решение:** дополнительную нормализацию громкости не включили в основной конвейер — убедительного практического выигрыша не получили.

**Ограничение:** сравнение выполнено при фиксированных порогах на конкретной выборке. Усиление изменяет одновременно речь и присутствующий шум, поэтому само по себе не улучшает акустический SNR.

## E12. Сравнение с WavLM

**Гипотеза:** другой готовый speaker encoder лучше перенесёт естественный шум.

Проверили microsoft/wavlm-base-plus-sv на тех же 102 validation-записях и трёх сценариях. Получали нормализованные эмбеддинги размерности 512, сравнивали cosine similarity и выбирали собственный порог модели. Версию весов зафиксировали.

In [ ]:
def wavlm_embedding_for_experiment(signal_16k, processor, model):
    import torch

    signal_16k = np.asarray(signal_16k)

    if (
        signal_16k.ndim != 1
        or signal_16k.dtype != np.dtype("float32")
        or not 16000 <= signal_16k.size <= 80000
        or not np.isfinite(signal_16k).all()
    ):
        raise ValueError(
            "Нужен конечный float32 mono-сигнал 16 кГц, 1–5 секунд"
        )

    if model.training:
        raise ValueError("Модель должна находиться в режиме eval")

    model_device = next(model.parameters()).device

    inputs = processor(
        signal_16k,
        sampling_rate=16000,
        return_tensors="pt",
    )
    inputs = {
        name: tensor.to(model_device)
        for name, tensor in inputs.items()
    }

    with torch.inference_mode():
        vector = model(**inputs).embeddings.reshape(-1)
        norm = vector.norm()

        if (
            vector.numel() != 512
            or not torch.isfinite(vector).all().item()
            or not torch.isfinite(norm).item()
            or norm.item() <= 1e-12
        ):
            raise RuntimeError("Некорректный эмбеддинг WavLM")

        return (vector / norm).cpu().numpy()


print("Логика извлечения эмбеддинга сохранена.")
print("Функция принимает уже загруженные доверенные model и processor.")
print("Скачивание, загрузка весов и инференс не выполнялись.")

In [ ]:
def summarize_wavlm_report():
    report = read_verified_report("wavlm")

    if (
        report["model"] != "microsoft/wavlm-base-plus-sv"
        or report["revision"]
        != "feb593a6c23c1cc3d9510425c29b0a14d2b07b1e"
    ):
        raise ValueError("Неожиданная модель или версия WavLM")

    threshold = report["threshold"]

    if (
        isinstance(threshold, bool)
        or not isinstance(threshold, (int, float))
        or not math.isfinite(threshold)
        or not -1 <= threshold <= 1
    ):
        raise ValueError("Некорректный порог")

    expected_scenarios = (
        "clean-clean",
        "clean-noisy",
        "noisy-noisy",
    )

    if set(report["scenarios"]) != set(expected_scenarios):
        raise ValueError("Неожиданные сценарии")

    errors = []

    for scenario in expected_scenarios:
        metrics = report["scenarios"][scenario]

        for key in ("far", "frr", "balanced_error"):
            value = metrics[key]

            if (
                isinstance(value, bool)
                or not isinstance(value, (int, float))
                or not math.isfinite(value)
                or not 0 <= value <= 1
            ):
                raise ValueError(f"Некорректная метрика: {scenario}/{key}")

        calculated_error = (metrics["far"] + metrics["frr"]) / 2

        if not math.isclose(
            calculated_error,
            metrics["balanced_error"],
            rel_tol=0,
            abs_tol=1e-12,
        ):
            raise ValueError("Не совпала ошибка сценария")

        errors.append(calculated_error)

    recorded_mean = report["balanced_error"]

    if (
        isinstance(recorded_mean, bool)
        or not isinstance(recorded_mean, (int, float))
        or not math.isfinite(recorded_mean)
        or not math.isclose(
            float(np.mean(errors)),
            recorded_mean,
            rel_tol=0,
            abs_tol=1e-12,
        )
    ):
        raise ValueError("Не совпала средняя ошибка")

    return report


wavlm_summary = None
wavlm_summary = summarize_wavlm_report()

print("СОХРАНЁННЫЙ VALIDATION — без повторного инференса")
print("Модель:", wavlm_summary["model"])
print("Версия:", wavlm_summary["revision"])
print(f"Порог: {wavlm_summary['threshold']:.6f}\n")

print(
    f"{'Сценарий':<16}"
    f"{'FAR':>10}"
    f"{'FRR':>10}"
    f"{'Ошибка':>10}"
)

for scenario in ("clean-clean", "clean-noisy", "noisy-noisy"):
    metrics = wavlm_summary["scenarios"][scenario]

    print(
        f"{scenario:<16}"
        f"{metrics['far']:>10.2%}"
        f"{metrics['frr']:>10.2%}"
        f"{metrics['balanced_error']:>10.2%}"
    )

print(
    f"\nСредняя ошибка WavLM: "
    f"{wavlm_summary['balanced_error']:.2%}"
)
print("Историческое сравнение на том же validation:")
print("ECAPA baseline: 2.68%")
print("ECAPA, ранняя адаптация на LibriSpeech: 2.49%")

print("\nХеш отчёта и согласованность метрик проверены.")
print("Новый порог не выбирался; test не использовался.")

**Результат:** WavLM получил среднюю validation-ошибку 11.60% против 2.68% у исходной ECAPA и 2.49% у ранней адаптации.

**Решение:** проверенную конфигурацию WavLM не включили в основной вариант. Продолжили адаптацию ECAPA под естественные clean/noisy условия.

**Ограничение:** вывод относится к конкретному checkpoint и конвейеру. Другие модели семейства WavLM и отдельное дообучение здесь не проверялись. Проверка версии в JSON не заменяет проверку самих весов при повторной загрузке.

## E13. Адаптация ECAPA на Vibravox

**Гипотеза:** обучение на естественных clean/noisy записях поможет сравнению одного голоса в разных условиях.

Использовали 144 обучающих диктора и 1654 записи: 827 clean и 827 noisy. Batch — четыре диктора, по одной clean/noisy записи каждого; фрагменты по три секунды. Обучали только fc и новую классификационную голову. Остальной encoder и статистики BatchNorm оставались замороженными.

In [ ]:
def create_vibravox_training_components(reference_encoder):
    import torch
    from copy import deepcopy

    # Передаётся доверенный исходный ECAPA, до адаптации.
    encoder = deepcopy(reference_encoder)
    encoder.requires_grad_(False)
    encoder.fc.requires_grad_(True)
    encoder.eval()

    # Одинаковая инициализация головы для сравниваемых вариантов.
    # Внешнее состояние CPU-генератора восстанавливается после блока.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(2031)

        head = torch.nn.Linear(
            in_features=192,
            out_features=144,
            bias=False,
        )
        torch.nn.init.xavier_uniform_(head.weight)

    model_device = next(encoder.parameters()).device
    head = head.to(model_device)
    head.train()

    optimizer = torch.optim.AdamW(
        [
            {"params": encoder.fc.parameters(), "lr": 1e-4},
            {"params": head.parameters(), "lr": 1e-3},
        ],
        weight_decay=1e-4,
    )

    if any(
        parameter.requires_grad
        for name, parameter in encoder.named_parameters()
        if not name.startswith("fc.")
    ):
        raise RuntimeError("Разморожены лишние параметры encoder")

    return encoder, head, optimizer


print("Логика создания обучаемых компонентов сохранена.")
print("Модель не создавалась, optimizer.step() не вызывался.")

Классификация использовала нормализованные эмбеддинги и веса головы, additive cosine margin 0.15 и scale 30.0. Градиенты ограничивали нормой 5.0. Validation выполняли каждые 100 шагов; checkpoint сохраняли при улучшении общей ошибки и допустимом качестве clean-clean.

In [ ]:
def validate_verification_metrics(evaluation):
    scenarios = ("clean-clean", "clean-noisy", "noisy-noisy")

    if set(evaluation["scenarios"]) != set(scenarios):
        raise ValueError("Неожиданные сценарии")

    threshold = evaluation["threshold"]

    if (
        isinstance(threshold, bool)
        or not isinstance(threshold, (int, float))
        or not math.isfinite(threshold)
        or not -1 <= threshold <= 1
    ):
        raise ValueError("Некорректный порог")

    errors = []

    for scenario in scenarios:
        metrics = evaluation["scenarios"][scenario]

        for key in ("far", "frr", "balanced_error"):
            value = metrics[key]

            if (
                isinstance(value, bool)
                or not isinstance(value, (int, float))
                or not math.isfinite(value)
                or not 0 <= value <= 1
            ):
                raise ValueError(f"Некорректная метрика: {scenario}/{key}")

        error = (metrics["far"] + metrics["frr"]) / 2

        if not math.isclose(
            error,
            metrics["balanced_error"],
            rel_tol=0,
            abs_tol=1e-12,
        ):
            raise ValueError("Не совпала ошибка сценария")

        errors.append(error)

    mean_error = evaluation["balanced_error"]

    if (
        isinstance(mean_error, bool)
        or not isinstance(mean_error, (int, float))
        or not math.isfinite(mean_error)
        or not math.isclose(
            float(np.mean(errors)),
            mean_error,
            rel_tol=0,
            abs_tol=1e-12,
        )
    ):
        raise ValueError("Не совпала средняя ошибка")


def summarize_first_300_steps():
    report = read_verified_report("classification_history")

    if (
        report["run_name"] != "vibravox_fc_classification_1200"
        or report["contrastive_weight"] != 0.0
    ):
        raise ValueError("Неожиданный вариант обучения")

    validate_verification_metrics(report["baseline"])

    first_steps = []

    for entry in report["history"]:
        if type(entry["step"]) is not int or entry["step"] <= 0:
            raise ValueError("Некорректный шаг обучения")

        if entry["step"] <= 300:
            validate_verification_metrics(entry["validation"])

            if type(entry["clean_ok"]) is not bool:
                raise ValueError("Некорректный признак clean_ok")

            first_steps.append(entry)

    if [entry["step"] for entry in first_steps] != [100, 200, 300]:
        raise ValueError("Не найдены ожидаемые контрольные шаги")

    return report["baseline"], first_steps


classification_300_summary = None
classification_300_summary = summarize_first_300_steps()

baseline_metrics, first_steps = classification_300_summary

print("СОХРАНЁННЫЙ VALIDATION — обучение не запускалось")
print("Первые 300 шагов сохранённого запуска на 1200 шагов.\n")

print(
    f"{'Шаг':<10}"
    f"{'Порог':>12}"
    f"{'Ошибка':>12}"
    f"{'Clean в допуске':>19}"
)

print(
    f"{'baseline':<10}"
    f"{baseline_metrics['threshold']:>12.6f}"
    f"{baseline_metrics['balanced_error']:>12.2%}"
    f"{'—':>19}"
)

for entry in first_steps:
    metrics = entry["validation"]

    print(
        f"{entry['step']:<10}"
        f"{metrics['threshold']:>12.6f}"
        f"{metrics['balanced_error']:>12.2%}"
        f"{str(entry['clean_ok']):>19}"
    )

print("\nХеш истории и согласованность метрик проверены.")
print("Показан флаг clean_ok из истории; замороженные веса")
print("и фактический ход обучения здесь повторно не проверялись.")
print("Test не использовался.")

**Результат:** на шаге 300 средняя validation-ошибка снизилась с 2.68% до 2.36%. Условие качества clean-clean выполнялось.

**Решение:** направление признали перспективным. Далее проверили дополнительный contrastive loss и увеличили бюджет, поскольку лучший результат пока находился на последнем проверенном шаге.

**Ограничение:** здесь показаны первые 300 шагов сохранённого расширенного запуска. Проверка истории не заменяет повторное обучение; данные о train-разбиении и конфигурации взяты из первоначального эксперимента.

## E14. Classification + contrastive loss

**Гипотеза:** дополнительный парный loss улучшит перенос между чистой и шумной речью одного человека.

Сравнили classification и classification + 0.1 × contrastive loss. Использовали одинаковые начальные веса и план batch: clean0, noisy0, clean1, noisy1 и далее. Для contrastive loss применили симметричную cross-entropy по матрице clean/noisy сходств с температурой 0.1.

In [ ]:
def vibravox_loss_components(vectors, head, targets):
    import torch
    import torch.nn.functional as functional

    if (
        vectors.shape != (8, 192)
        or not vectors.is_floating_point()
        or targets.shape != (8,)
        or targets.dtype != torch.long
        or head.weight.shape != (144, 192)
        or head.bias is not None
    ):
        raise ValueError("Неожиданные формы эмбеддингов, головы или меток")

    if (
        vectors.device != targets.device
        or vectors.device != head.weight.device
        or vectors.dtype != head.weight.dtype
    ):
        raise ValueError("Не совпали устройства или типы тензоров")

    if (
        not torch.isfinite(vectors).all().item()
        or not torch.isfinite(head.weight).all().item()
        or torch.any(vectors.norm(dim=1) <= 1e-12).item()
        or torch.any(head.weight.norm(dim=1) <= 1e-12).item()
    ):
        raise ValueError("Некорректные эмбеддинги или веса головы")

    if (
        targets.min().item() < 0
        or targets.max().item() >= 144
        or not torch.equal(targets[0::2], targets[1::2])
        or targets[0::2].unique().numel() != 4
    ):
        raise ValueError("Ожидались четыре разных clean/noisy пары")

    normalized = functional.normalize(vectors, dim=1)
    normalized_weights = functional.normalize(head.weight, dim=1)

    cosine_logits = functional.linear(
        normalized,
        normalized_weights,
    )

    target_mask = functional.one_hot(
        targets,
        num_classes=144,
    ).to(cosine_logits.dtype)

    classification_loss = functional.cross_entropy(
        30.0 * (cosine_logits - 0.15 * target_mask),
        targets,
    )

    clean_vectors = normalized[0::2]
    noisy_vectors = normalized[1::2]

    cross_logits = clean_vectors @ noisy_vectors.T / 0.1
    matching_targets = torch.arange(4, device=vectors.device)

    contrastive_loss = 0.5 * (
        functional.cross_entropy(cross_logits, matching_targets)
        + functional.cross_entropy(cross_logits.T, matching_targets)
    )

    if not (
        torch.isfinite(classification_loss).item()
        and torch.isfinite(contrastive_loss).item()
    ):
        raise RuntimeError("Получен неконечный loss")

    return classification_loss, contrastive_loss


print("Логика classification и contrastive loss сохранена.")
print("Градиенты и обновления весов не вычислялись.")
print("Порядок clean/noisy должен обеспечиваться планом batch;")
print("совпадение меток само по себе акустические условия не проверяет.")

In [ ]:
def compare_loss_histories_first_300():
    classification = read_verified_report("classification_history")
    contrastive = read_verified_report("contrastive_history")

    for report, expected_name, expected_weight in (
        (
            classification,
            "vibravox_fc_classification_1200",
            0.0,
        ),
        (
            contrastive,
            "vibravox_fc_contrastive_1200",
            0.1,
        ),
    ):
        if (
            report["run_name"] != expected_name
            or report["contrastive_weight"] != expected_weight
        ):
            raise ValueError("Неожиданная конфигурация запуска")

        validate_verification_metrics(report["baseline"])

    if classification["baseline"] != contrastive["baseline"]:
        raise ValueError("У запусков разные baseline-результаты")

    histories = []

    for report in (classification, contrastive):
        selected = {}

        for entry in report["history"]:
            step = entry["step"]

            if type(step) is not int or step <= 0:
                raise ValueError("Некорректный шаг")

            if step <= 300:
                if step in selected:
                    raise ValueError("Повторный контрольный шаг")

                validate_verification_metrics(entry["validation"])
                selected[step] = entry["validation"]

        if set(selected) != {100, 200, 300}:
            raise ValueError("Не хватает контрольных шагов")

        histories.append(selected)

    return histories


loss_comparison_300 = None
loss_comparison_300 = compare_loss_histories_first_300()

classification_steps, contrastive_steps = loss_comparison_300

print("СОХРАНЁННЫЙ VALIDATION — без повторного обучения\n")
print(
    f"{'Шаг':<8}"
    f"{'Classification':>17}"
    f"{'+ Contrastive':>17}"
    f"{'Δ, п.п.':>12}"
)

for step in (100, 200, 300):
    class_error = classification_steps[step]["balanced_error"]
    pair_error = contrastive_steps[step]["balanced_error"]

    print(
        f"{step:<8}"
        f"{class_error:>17.2%}"
        f"{pair_error:>17.2%}"
        f"{100 * (pair_error - class_error):>+12.3f}"
    )

print("\nОтрицательная Δ означает меньшую ошибку варианта с contrastive.")
print("Использованы первые 300 шагов сохранённых расширенных запусков.")
print("Хеши историй и согласованность метрик проверены.")
print("Одинаковость исходных весов и batch-плана здесь не перепроверялась.")
print("Test не использовался.")

**Результат:** на шаге 300 classification дал 2.36%, classification + contrastive — 2.35%. Дополнительный выигрыш оказался небольшим.

**Решение:** contrastive loss не приняли в основной вариант по этому результату. Оба варианта продолжили до 1200 шагов, чтобы проверить зависимость от длительности обучения.

**Ограничение:** сравнение относится к весу 0.1, температуре 0.1 и конкретному batch-плану. Оно не доказывает бесполезность других парных losses или параметров.

## E15. Увеличение обучения до 1200 шагов

**Гипотеза:** результат на шаге 300 мог быть ограничен слишком ранней остановкой.

Оба варианта обучали до 1200 шагов с validation каждые 100 шагов. Лучший checkpoint обновляли при снижении средней ошибки более чем на 0.000001 и допустимой clean-clean ошибке: не выше baseline + 2 процентных пункта. Последние веса сохраняли отдельно от лучших.

In [ ]:
def validate_extended_training_history(report_name, run_name, loss_weight):
    report = read_verified_report(report_name)

    if (
        report["run_name"] != run_name
        or report["contrastive_weight"] != loss_weight
    ):
        raise ValueError("Неожиданный вариант обучения")

    validate_verification_metrics(report["baseline"])

    history = report["history"]

    if not isinstance(history, list) or len(history) != 12:
        raise ValueError("Ожидалось 12 контрольных шагов")

    steps = [entry["step"] for entry in history]

    if (
        any(type(step) is not int for step in steps)
        or steps != list(range(100, 1201, 100))
    ):
        raise ValueError("Нарушен порядок контрольных шагов")

    clean_limit = (
        report["baseline"]["scenarios"]["clean-clean"]["balanced_error"]
        + 0.02
    )

    best_step = 0
    best_validation = report["baseline"]

    for entry in history:
        validation = entry["validation"]
        validate_verification_metrics(validation)

        clean_ok = (
            validation["scenarios"]["clean-clean"]["balanced_error"]
            <= clean_limit
        )

        if (
            type(entry["clean_ok"]) is not bool
            or entry["clean_ok"] != clean_ok
        ):
            raise ValueError("Не совпало условие качества clean-clean")

        improved = (
            validation["balanced_error"]
            < best_validation["balanced_error"] - 1e-6
        )

        if improved and clean_ok:
            best_step = entry["step"]
            best_validation = validation

    validate_verification_metrics(report["best_validation"])

    if (
        type(report["best_step"]) is not int
        or report["best_step"] != best_step
        or report["best_validation"] != best_validation
    ):
        raise ValueError("Сохранённый лучший checkpoint выбран иначе")

    return report


def load_extended_training_comparison():
    classification = validate_extended_training_history(
        "classification_history",
        "vibravox_fc_classification_1200",
        0.0,
    )
    contrastive = validate_extended_training_history(
        "contrastive_history",
        "vibravox_fc_contrastive_1200",
        0.1,
    )

    if classification["baseline"] != contrastive["baseline"]:
        raise ValueError("У запусков разные baseline-результаты")

    return {
        "classification": classification,
        "classification_contrastive": contrastive,
    }


extended_training_summary = None
extended_training_summary = load_extended_training_comparison()

classification_history = extended_training_summary[
    "classification"
]["history"]

contrastive_history = extended_training_summary[
    "classification_contrastive"
]["history"]

print("СОХРАНЁННЫЙ VALIDATION — без повторного обучения\n")
print(
    f"{'Шаг':<8}"
    f"{'Classification':>17}"
    f"{'+ Contrastive':>17}"
)

for classification_entry, contrastive_entry in zip(
    classification_history,
    contrastive_history,
):
    print(
        f"{classification_entry['step']:<8}"
        f"{classification_entry['validation']['balanced_error']:>17.2%}"
        f"{contrastive_entry['validation']['balanced_error']:>17.2%}"
    )

print("\nЛучшие checkpoint по восстановленному правилу выбора:")
print(
    f"{'Вариант':<29}"
    f"{'Шаг':>7}"
    f"{'Ошибка':>12}"
    f"{'Порог':>12}"
)

for name, report in extended_training_summary.items():
    best = report["best_validation"]

    print(
        f"{name:<29}"
        f"{report['best_step']:>7}"
        f"{best['balanced_error']:>12.2%}"
        f"{best['threshold']:>12.6f}"
    )

print("\nПравило выбора checkpoint воспроизведено по истории.")
print("Хеши отчётов, метрики и флаги clean_ok проверены.")
print("Содержимое весов и выполнение обучения здесь не проверялись.")
print("Test не использовался.")

**Результат:** у обоих вариантов лучший checkpoint оказался на шаге 500. Classification получил около 2.10%, classification + contrastive — 2.09%. Дальнейшее обучение не улучшило лучший результат по выбранному критерию.

**Решение:** оставили classification checkpoint шага 500 как более простой вариант. Дополнительный contrastive loss дал около 0.007 процентного пункта выигрыша — недостаточно для убедительного преимущества без независимого подтверждения.

**Ограничение:** ухудшение validation после лучшего шага согласуется с началом переобучения или колебаниями обучения, но одна история запуска не позволяет однозначно определить причину. Выбор лучшего шага выполнялся по validation; перенос выбранной модели далее проверили на 1000 test-записях.

## E16. Проверка на 1000 test-записях

**Гипотеза:** улучшение на validation переносится на большее количество высказываний.

Проверили 1000 записей от 20 test-дикторов: 847 clean и 153 noisy. Сравнили исходную ECAPA и classification-checkpoint шага 500. Для каждой модели использовали собственный заранее выбранный validation-порог; на test пороги не подбирали.

In [ ]:
def summarize_test_1000():
    report = read_verified_report("test_1000")

    if (
        type(report["records"]) is not int
        or report["records"] != 1000
        or type(report["speakers"]) is not int
        or report["speakers"] != 20
        or type(report["step"]) is not int
        or report["step"] != 500
    ):
        raise ValueError("Неожиданный размер test или шаг checkpoint")

    expected_checkpoint = (
        project_dir
        / "checkpoints/vibravox_fc_classification_1200/best.pt"
    ).resolve()

    if Path(report["checkpoint"]).resolve() != expected_checkpoint:
        raise ValueError("В отчёте указан другой checkpoint")

    models = ("baseline", "trained")
    scenarios = ("clean-clean", "clean-noisy", "noisy-noisy")

    if (
        set(report["thresholds"]) != set(models)
        or set(report["mean_errors"]) != set(models)
        or set(report["scenarios"]) != set(models)
    ):
        raise ValueError("Неожиданный набор моделей")

    training_report = extended_training_summary["classification"]

    expected_thresholds = {
        "baseline": training_report["baseline"]["threshold"],
        "trained": training_report["best_validation"]["threshold"],
    }

    for model in models:
        threshold = report["thresholds"][model]

        if (
            isinstance(threshold, bool)
            or not isinstance(threshold, (int, float))
            or not math.isfinite(threshold)
            or not math.isclose(
                threshold,
                expected_thresholds[model],
                rel_tol=0,
                abs_tol=1e-12,
            )
        ):
            raise ValueError("Порог не соответствует validation-истории")

        validate_verification_metrics({
            "threshold": threshold,
            "balanced_error": report["mean_errors"][model],
            "scenarios": report["scenarios"][model],
        })

        pair_count = 0

        for scenario in scenarios:
            metrics = report["scenarios"][model][scenario]

            for key in (
                "same_count",
                "different_count",
                "false_same",
                "false_different",
            ):
                if type(metrics[key]) is not int:
                    raise ValueError("Счётчики должны быть целыми")

            same_count = metrics["same_count"]
            different_count = metrics["different_count"]

            if not (
                same_count > 0
                and different_count > 0
                and 0 <= metrics["false_same"] <= different_count
                and 0 <= metrics["false_different"] <= same_count
            ):
                raise ValueError("Некорректные счётчики")

            if not (
                math.isclose(
                    metrics["far"],
                    metrics["false_same"] / different_count,
                    rel_tol=0,
                    abs_tol=1e-12,
                )
                and math.isclose(
                    metrics["frr"],
                    metrics["false_different"] / same_count,
                    rel_tol=0,
                    abs_tol=1e-12,
                )
            ):
                raise ValueError("Метрики не соответствуют счётчикам")

            pair_count += same_count + different_count

        expected_pair_count = (
            report["records"] * (report["records"] - 1) // 2
        )

        if pair_count != expected_pair_count:
            raise ValueError("Не совпало общее число пар")

    for scenario in scenarios:
        baseline = report["scenarios"]["baseline"][scenario]
        trained = report["scenarios"]["trained"][scenario]

        if (
            baseline["same_count"] != trained["same_count"]
            or baseline["different_count"] != trained["different_count"]
        ):
            raise ValueError("У моделей разные числа пар")

    return report


test_1000_summary = None
test_1000_summary = summarize_test_1000()

print("СОХРАНЁННЫЙ TEST — без повторного инференса")
print("Записей:", test_1000_summary["records"])
print("Дикторов:", test_1000_summary["speakers"])
print("Шаг trained:", test_1000_summary["step"])
print("Пороги:", test_1000_summary["thresholds"], "\n")

print(
    f"{'Сценарий':<15}{'Модель':<10}"
    f"{'SAME':>8}{'DIFF':>10}"
    f"{'FAR':>10}{'FRR':>10}{'Ошибка':>10}"
)

for scenario in ("clean-clean", "clean-noisy", "noisy-noisy"):
    for model in ("baseline", "trained"):
        metrics = test_1000_summary["scenarios"][model][scenario]

        print(
            f"{scenario:<15}{model:<10}"
            f"{metrics['same_count']:>8}"
            f"{metrics['different_count']:>10}"
            f"{metrics['far']:>10.2%}"
            f"{metrics['frr']:>10.2%}"
            f"{metrics['balanced_error']:>10.2%}"
        )

print("\nСредняя ошибка — равный вес трёх сценариев:")

for model in ("baseline", "trained"):
    print(
        f"{model:<10}: "
        f"{test_1000_summary['mean_errors'][model]:.2%}"
    )

delta = (
    test_1000_summary["mean_errors"]["trained"]
    - test_1000_summary["mean_errors"]["baseline"]
)

print(f"Изменение trained − baseline: {100 * delta:+.3f} п.п.")
print("\nХеш отчёта, пороги, счётчики и формулы проверены.")
print("Совпадение идентичности всех пар здесь не перепроверялось.")
print("Пороги на test не подбирались.")

**Результат:** средняя ошибка снизилась с 3.26% до 3.07%. В clean-noisy ошибка уменьшилась с 5.97% до 4.72%, но noisy-noisy ухудшился с 2.27% до 2.88%. В clean-clean ошибка практически не изменилась.

**Решение:** сохранили trained как кандидат, полезный для clean-noisy, но не объявили универсально лучшим. Следующим шагом проверили статистическую устойчивость различий.

**Ограничение:** 1000 записей принадлежат тем же 20 test-дикторам — это расширение числа высказываний, а не независимая группа новых людей. Пары зависимы, поскольку используют общие записи и дикторов.

## E17. Bootstrap по дикторам

**Гипотеза:** наблюдаемое улучшение может зависеть от состава test-дикторов.

Выполнили 5000 перевыборок 20 дикторов с возвращением, seed 2034. Одинаковые перевыборки использовали для обеих моделей и всех сценариев. Пороги оставались фиксированными.

SAME-пары взвешивали кратностью одного диктора, DIFFERENT-пары — произведением кратностей двух дикторов. Для разности ошибок trained − baseline рассчитали процентильный 95% интервал.

In [ ]:
def summarize_speaker_bootstrap():
    report = read_verified_report("bootstrap")

    if (
        type(report["seed"]) is not int
        or report["seed"] != 2034
        or type(report["repeats"]) is not int
        or report["repeats"] != 5000
        or type(report["speakers"]) is not int
        or report["speakers"] != 20
    ):
        raise ValueError("Неожиданные параметры bootstrap")

    if report["fixed_thresholds"] != test_1000_summary["thresholds"]:
        raise ValueError("Пороги отличаются от test-оценки")

    expected_names = (
        "clean-clean",
        "clean-noisy",
        "noisy-noisy",
        "mean",
    )

    if set(report["results"]) != set(expected_names):
        raise ValueError("Неожиданные результаты bootstrap")

    for name in expected_names:
        result = report["results"][name]

        for key in (
            "observed_delta_pp",
            "ci95_lower_pp",
            "ci95_upper_pp",
        ):
            value = result[key]

            if (
                isinstance(value, bool)
                or not isinstance(value, (int, float))
                or not math.isfinite(value)
                or not -100 <= value <= 100
            ):
                raise ValueError("Некорректные значения интервала")

        if result["ci95_lower_pp"] > result["ci95_upper_pp"]:
            raise ValueError("Перепутаны границы интервала")

        if (
            type(result["valid_repeats"]) is not int
            or not 1 <= result["valid_repeats"] <= report["repeats"]
        ):
            raise ValueError("Некорректное число допустимых повторов")

        if name == "mean":
            expected_delta = 100 * (
                test_1000_summary["mean_errors"]["trained"]
                - test_1000_summary["mean_errors"]["baseline"]
            )
        else:
            expected_delta = 100 * (
                test_1000_summary["scenarios"]["trained"][name][
                    "balanced_error"
                ]
                - test_1000_summary["scenarios"]["baseline"][name][
                    "balanced_error"
                ]
            )

        if not math.isclose(
            result["observed_delta_pp"],
            expected_delta,
            rel_tol=0,
            abs_tol=1e-10,
        ):
            raise ValueError("Разность не совпала с test-отчётом")

    return report


speaker_bootstrap_summary = None
speaker_bootstrap_summary = summarize_speaker_bootstrap()

print("СОХРАНЁННЫЙ BOOTSTRAP — без повторной перевыборки\n")
print(
    f"{'Сценарий':<16}"
    f"{'Δ, п.п.':>12}"
    f"{'95% интервал, п.п.':>27}"
    f"{'Повторов':>11}"
)

for name in ("clean-clean", "clean-noisy", "noisy-noisy", "mean"):
    result = speaker_bootstrap_summary["results"][name]

    interval = (
        f"[{result['ci95_lower_pp']:+.3f}; "
        f"{result['ci95_upper_pp']:+.3f}]"
    )

    print(
        f"{name:<16}"
        f"{result['observed_delta_pp']:>+12.3f}"
        f"{interval:>27}"
        f"{result['valid_repeats']:>11}"
    )

print("\nОтрицательная Δ — улучшение trained.")
print("Положительная Δ — ухудшение trained.")
print("Хеш отчёта и соответствие наблюдаемой Δ test-метрикам проверены.")
print("Границы интервалов здесь повторно не рассчитывались.")

**Результат:** общая разность составила −0.197 п.п., 95% интервал [−0.487; +0.140] включает ноль. Для clean-noisy интервал полностью отрицательный, для noisy-noisy — полностью положительный.

**Решение:** не заявляем подтверждённого общего превосходства trained. В рамках этой процедуры улучшение clean-noisy и ухудшение noisy-noisy устойчивее общей разницы.

**Ограничение:** bootstrap условен на этих 20 дикторах, фиксированных порогах и выбранной процедуре. Он не устраняет последствия многократного просмотра test и не заменяет проверку на новых отложенных дикторах.

## E18. Длительность и положение фрагмента

**Гипотеза:** короткие записи и начало с паузой ухудшают верификацию.

Из одинаковых записей длительностью не менее пяти секунд брали фрагменты 1, 2, 3 и 5 секунд. Дополнительно сравнили начало и центр. В test-поднабор вошли 383 записи от 20 дикторов, в validation — 36 записей от 12 дикторов. Пороги оставались фиксированными.

In [ ]:
def crop_duration_fragment(signal_16k, seconds, position="first"):
    signal = np.asarray(signal_16k)

    if (
        signal.shape != (80000,)
        or signal.dtype != np.dtype("float32")
        or not np.isfinite(signal).all()
    ):
        raise ValueError("Ожидалась подготовленная запись 5 с, 16 кГц")

    if type(seconds) is not int or seconds not in (1, 2, 3, 5):
        raise ValueError("Допустимая длительность: 1, 2, 3 или 5 секунд")

    if position not in ("first", "center"):
        raise ValueError("Допустимое положение: first или center")

    length = seconds * 16000
    start = 0 if position == "first" else (signal.size - length) // 2

    return signal[start:start + length].copy()


def load_duration_comparison():
    first = read_verified_report("duration")
    center = read_verified_report("duration_center")

    for report in (first, center):
        if report["thresholds"] != test_1000_summary["thresholds"]:
            raise ValueError("Пороги отличаются от основной test-оценки")

    if set(first["results"]) != {"validation", "test"}:
        raise ValueError("Неожиданные разбиения")

    for split in ("validation", "test"):
        if set(first["results"][split]) != {"1", "2", "3", "5"}:
            raise ValueError("Неожиданные длительности")

        for duration, models in first["results"][split].items():
            if set(models) != {"baseline", "trained"}:
                raise ValueError("Неожиданные модели")

            for model, metrics in models.items():
                validate_verification_metrics({
                    "threshold": first["thresholds"][model],
                    "balanced_error": metrics["mean_error"],
                    "scenarios": metrics["scenarios"],
                })

    if set(center["results"]) != {"1", "2", "3"}:
        raise ValueError("Неожиданные центральные фрагменты")

    for duration, models in center["results"].items():
        if set(models) != {"baseline", "trained"}:
            raise ValueError("Неожиданные модели")

        for model, metrics in models.items():
            validate_verification_metrics({
                "threshold": center["thresholds"][model],
                "balanced_error": metrics["mean_error"],
                "scenarios": metrics["scenarios"],
            })

    return first, center


duration_summary = None
duration_summary = load_duration_comparison()
duration_first, duration_center = duration_summary

print("СОХРАНЁННАЯ ОЦЕНКА — без повторной нарезки и инференса\n")
print(
    f"{'Группа':<13}{'Секунд':>7}{'Модель':>12}"
    f"{'Начало':>12}{'Центр':>12}"
)

for split in ("validation", "test"):
    for seconds in ("1", "2", "3", "5"):
        for model in ("baseline", "trained"):
            first_error = duration_first["results"][split][seconds][
                model
            ]["mean_error"]

            if split == "test" and seconds != "5":
                center_error = duration_center["results"][seconds][
                    model
                ]["mean_error"]
                center_text = f"{center_error:.2%}"
            else:
                center_text = "—"

            print(
                f"{split:<13}{seconds:>7}{model:>12}"
                f"{first_error:>12.2%}{center_text:>12}"
            )

print("\nХеши, фиксированные пороги и формулы метрик проверены.")
print("Центральные результаты в этой таблице показаны только для test.")
print("Для 5 секунд используется вся подготовленная запись.")

**Результат:** у trained ошибка на test для начала записи составила 42.13%, 14.54%, 5.51% и 2.65% для 1/2/3/5 секунд. Центральные фрагменты улучшили результаты 1–3 секунд, но не устранили проблему короткой речи.

**Решение:** рекомендовать 3–5 секунд речи. Автоматическую центральную обрезку в основной конвейер не включали.

**Ограничение:** отдельные пороги для каждой длительности не подбирали. Поднаборы меньше исходных, особенно validation; выводы относятся к этому рабочему режиму.

## E19. Рабочие пороги и цена ошибок

**Гипотеза:** порог минимальной средней ошибки может не подходить требованиям к ложным совпадениям.

На validation выбрали пороги для среднего FAR 1%, 2% и 5%. Режим 1% проверили на test без дополнительного подбора.

Для итогового интерфейса отдельно реализовали минимизацию стоимости ошибок: C_FA × P(DIFFERENT) × FAR + C_FR × P(SAME) × FRR. По умолчанию цены равны, доли классов — 50/50; ползунок меняет соотношение цен.

In [ ]:
def check_rate(value):
    if (
        isinstance(value, bool)
        or not isinstance(value, (int, float))
        or not math.isfinite(value)
        or not 0 <= value <= 1
    ):
        raise ValueError("Некорректная вероятность или доля ошибок")


def load_operating_comparison():
    candidates = read_verified_report("target_far_validation")
    test = read_verified_report("target_far_test")

    if set(candidates) != {"baseline", "trained"}:
        raise ValueError("Неожиданные модели")

    for model in ("baseline", "trained"):
        if set(candidates[model]) != {"0.01", "0.02", "0.05"}:
            raise ValueError("Неожиданные цели FAR")

        for target, point in candidates[model].items():
            if point["target_mean_far"] != float(target):
                raise ValueError("Не совпала цель FAR")

            if set(point["scenarios"]) != {
                "clean-clean", "clean-noisy", "noisy-noisy"
            }:
                raise ValueError("Неожиданные сценарии")

            for key in (
                "mean_far", "mean_frr", "worst_scenario_far"
            ):
                check_rate(point[key])

            far_values = []
            frr_values = []

            for metrics in point["scenarios"].values():
                check_rate(metrics["far"])
                check_rate(metrics["frr"])
                far_values.append(metrics["far"])
                frr_values.append(metrics["frr"])

            if not (
                math.isclose(
                    point["mean_far"], float(np.mean(far_values)),
                    rel_tol=0, abs_tol=1e-12,
                )
                and math.isclose(
                    point["mean_frr"], float(np.mean(frr_values)),
                    rel_tol=0, abs_tol=1e-12,
                )
                and math.isclose(
                    point["worst_scenario_far"], max(far_values),
                    rel_tol=0, abs_tol=1e-12,
                )
                and point["mean_far"] <= float(target) + 1e-12
            ):
                raise ValueError("Не согласованы validation-метрики")

            threshold = point["threshold"]

            if (
                isinstance(threshold, bool)
                or not isinstance(threshold, (int, float))
                or not math.isfinite(threshold)
                or not -1 <= threshold <= 1
            ):
                raise ValueError("Некорректный порог")

    if test["target_validation_mean_far"] != 0.01:
        raise ValueError("Ожидалась test-проверка режима FAR 1%")

    if (
        set(test["thresholds"]) != {"baseline", "trained"}
        or set(test["test"]) != {"baseline", "trained"}
    ):
        raise ValueError("Неожиданные test-модели")

    for model in ("baseline", "trained"):
        if test["thresholds"][model] != candidates[model]["0.01"]["threshold"]:
            raise ValueError("На test использован другой порог")

        metrics = test["test"][model]
        mean_error = float(np.mean([
            item["balanced_error"] for item in metrics.values()
        ]))

        validate_verification_metrics({
            "threshold": test["thresholds"][model],
            "balanced_error": mean_error,
            "scenarios": metrics,
        })

    return candidates, test


operating_comparison = None
operating_comparison = load_operating_comparison()
operating_candidates, target_far_test = operating_comparison

print("СОХРАНЁННЫЙ VALIDATION — пороги заново не подбирались\n")
print(
    f"{'Модель':<10}{'Цель FAR':>10}{'Порог':>12}"
    f"{'FAR сред.':>12}{'FRR сред.':>12}{'FAR худш.':>12}"
)

for model in ("baseline", "trained"):
    for target in ("0.01", "0.02", "0.05"):
        point = operating_candidates[model][target]

        print(
            f"{model:<10}{float(target):>10.0%}"
            f"{point['threshold']:>12.6f}"
            f"{point['mean_far']:>12.2%}"
            f"{point['mean_frr']:>12.2%}"
            f"{point['worst_scenario_far']:>12.2%}"
        )

print("\nTEST при цели среднего validation FAR 1%:")
print(f"{'Модель':<10}{'FAR сред.':>12}{'FRR сред.':>12}")

for model in ("baseline", "trained"):
    metrics = target_far_test["test"][model]

    mean_far = float(np.mean([item["far"] for item in metrics.values()]))
    mean_frr = float(np.mean([item["frr"] for item in metrics.values()]))

    print(f"{model:<10}{mean_far:>12.2%}{mean_frr:>12.2%}")

print("\nХеши, метрики и перенос порогов validation → test проверены.")

**Результат:** при цели среднего validation FAR 1% на test получены FAR/FRR 1.05%/8.33% у baseline и 0.82%/8.27% у trained.

**Решение:** сохранять рабочий порог вместе с критерием его выбора. По уточнённому требованию основной режим использует одинаковую цену ошибок; режимы с ограничением FAR остаются отдельными исследованными вариантами.

**Ограничение:** средний FAR 1% не гарантирует FAR ≤1% в каждом сценарии или на новых данных. Частоту SAME-запросов нельзя оценивать по искусственному соотношению пар датасета.

## E20. Калибровка confidence

**Гипотеза:** similarity можно преобразовать в более интерпретируемую оценку p_same.

На validation обучили логистический калибратор, не меняя ECAPA. При обучении и оценке использовали равные веса классов и сценариев. На test проверили Brier score, log loss и ECE по десяти интервалам.

Confidence равен p_same для ответа SAME и 1 − p_same для DIFFERENT. Изменение цены ошибок меняет порог, а не p_same одной пары.

In [ ]:
def load_calibration_summary():
    report = read_verified_report("calibration")

    if (
        type(report["step"]) is not int
        or report["step"] != 500
        or report["assumed_same_prior"] != 0.5
        or report["equal_scenario_weights"] is not True
    ):
        raise ValueError("Неожиданная конфигурация калибратора")

    if Path(report["checkpoint"]).resolve() != Path(
        test_1000_summary["checkpoint"]
    ).resolve():
        raise ValueError("Калибратор относится к другой модели")

    for key in ("a", "b", "decision_threshold"):
        value = report[key]

        if (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
            or not math.isfinite(value)
        ):
            raise ValueError("Некорректные параметры калибратора")

    if (
        report["a"] <= 0
        or report["decision_threshold"]
        != test_1000_summary["thresholds"]["trained"]
    ):
        raise ValueError("Некорректный наклон или несовместимый порог")

    metrics = report["test_metrics"]
    check_rate(metrics["brier"])
    check_rate(metrics["ece_10_bins"])

    log_loss = metrics["log_loss"]

    if (
        isinstance(log_loss, bool)
        or not isinstance(log_loss, (int, float))
        or not math.isfinite(log_loss)
        or log_loss < 0
    ):
        raise ValueError("Некорректный log loss")

    bins = report["reliability"]

    if not isinstance(bins, list) or len(bins) != 10:
        raise ValueError("Ожидалось десять заполненных интервалов")

    for index, row in enumerate(bins):
        for key in ("lower", "upper", "predicted", "observed", "weight"):
            check_rate(row[key])

        if not (
            math.isclose(row["lower"], index / 10, abs_tol=1e-12)
            and math.isclose(row["upper"], (index + 1) / 10, abs_tol=1e-12)
            and row["lower"] <= row["predicted"] <= row["upper"]
        ):
            raise ValueError("Некорректные границы интервала")

    if not math.isclose(
        sum(row["weight"] for row in bins),
        1.0,
        rel_tol=0,
        abs_tol=1e-12,
    ):
        raise ValueError("Веса интервалов не суммируются в единицу")

    calculated_ece = sum(
        row["weight"] * abs(row["predicted"] - row["observed"])
        for row in bins
    )

    if not math.isclose(
        calculated_ece,
        metrics["ece_10_bins"],
        rel_tol=0,
        abs_tol=1e-12,
    ):
        raise ValueError("ECE не соответствует сводкам интервалов")

    return report


calibration_summary = None
calibration_summary = load_calibration_summary()

print("СОХРАНЁННАЯ ОЦЕНКА КАЛИБРОВКИ НА TEST\n")
print(f"a: {calibration_summary['a']:.6f}")
print(f"b: {calibration_summary['b']:.6f}")

for name, value in calibration_summary["test_metrics"].items():
    print(f"{name}: {value:.6f}")

print(
    f"\n{'Интервал':<12}{'Средний p':>12}"
    f"{'Доля SAME':>12}{'Вес':>12}"
)

for row in calibration_summary["reliability"]:
    label = f"{row['lower']:.1f}–{row['upper']:.1f}"

    print(
        f"{label:<12}"
        f"{row['predicted']:>12.3f}"
        f"{row['observed']:>12.3f}"
        f"{row['weight']:>12.2%}"
    )

print("\nЛогика преобразования similarity → p_same:")

for score in (
    0.0,
    calibration_summary["decision_threshold"],
    0.5,
    0.8,
):
    logit = (
        calibration_summary["a"] * score
        + calibration_summary["b"]
    )

    # Численно устойчивый sigmoid.
    probability = float(np.exp(-np.logaddexp(0.0, -logit)))

    print(f"{score:.6f} → {probability:.6f}")

print("\nХеш, согласованность модели и ECE по сводкам проверены.")
print("Brier и log loss без исходных предсказаний не пересчитывались.")
print("Калибратор повторно не обучался.")

**Результат:** на test получены Brier 0.0226, log loss 0.0766 и ECE 0.0089 при балансировке классов и сценариев.

**Решение:** приняли калибратор как предварительный способ выдачи confidence. Порог решения сохраняется отдельно: при неравных ценах ошибок confidence выданного ответа может быть ниже 0.5.

**Ограничение:** эти показатели не гарантируют качество вероятностей для другого домена, иной доли SAME-запросов или короткой речи. ECE зависит от разбиения на интервалы и не является достаточной проверкой сам по себе.

## E21. ECAPA + logistic regression / SVM

**Гипотеза:** обучаемый классификатор пары использует эмбеддинги лучше cosine.

Для двух эмбеддингов формировали признаки abs(e1 − e2) и e1 × e2: всего 384 признака, симметричных относительно перестановки записей. Использовали 11970 train-пар с балансом классов и сценариев. StandardScaler и классификатор обучали только на train; C и порог выбирали на validation.

In [ ]:
def symmetric_pair_features(first, second):
    first = np.asarray(first)
    second = np.asarray(second)

    if (
        first.ndim != 2
        or first.shape != second.shape
        or first.shape[1] not in (40, 192)
        or not 1 <= first.shape[0] <= 12000
        or first.dtype != np.dtype("float32")
        or second.dtype != np.dtype("float32")
        or not np.isfinite(first).all()
        or not np.isfinite(second).all()
    ):
        raise ValueError("Некорректные матрицы признаков записей")

    with np.errstate(over="raise", invalid="raise"):
        result = np.concatenate(
            [np.abs(first - second), first * second],
            axis=1,
        )

    if not np.isfinite(result).all():
        raise RuntimeError("Некорректные признаки пары")

    return result


def make_pair_classifier(method, c_value):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC

    if (
        isinstance(c_value, bool)
        or c_value not in (0.01, 0.1, 1.0)
    ):
        raise ValueError("Неожиданное значение C")

    if method == "logistic":
        classifier = LogisticRegression(
            C=c_value,
            solver="lbfgs",
            max_iter=2000,
            random_state=2036,
        )
    elif method == "svm":
        classifier = LinearSVC(
            C=c_value,
            dual=False,
            max_iter=20000,
            random_state=2036,
        )
    else:
        raise ValueError("Неизвестный классификатор")

    return make_pipeline(StandardScaler(), classifier)


print("Логика признаков и создания классификаторов сохранена.")
print("Обучение не выполнялось; сериализованные модели не загружались.")

In [ ]:
def validate_backend_evaluation(evaluation):
    threshold = evaluation["threshold"]

    if (
        isinstance(threshold, bool)
        or not isinstance(threshold, (int, float))
        or not math.isfinite(threshold)
    ):
        raise ValueError("Некорректный порог backend")

    scenarios = evaluation["scenarios"]

    if set(scenarios) != {"clean-clean", "clean-noisy", "noisy-noisy"}:
        raise ValueError("Неожиданные сценарии")

    far_values = []
    frr_values = []

    for metrics in scenarios.values():
        for key in ("far", "frr", "balanced_error"):
            check_rate(metrics[key])

        if not math.isclose(
            metrics["balanced_error"],
            (metrics["far"] + metrics["frr"]) / 2,
            rel_tol=0,
            abs_tol=1e-12,
        ):
            raise ValueError("Не совпала ошибка сценария")

        far_values.append(metrics["far"])
        frr_values.append(metrics["frr"])

    calculated = {
        "mean_far": float(np.mean(far_values)),
        "mean_frr": float(np.mean(frr_values)),
        "balanced_error": float(
            (np.mean(far_values) + np.mean(frr_values)) / 2
        ),
    }

    for key, expected in calculated.items():
        if key == "balanced_error" or key in evaluation:
            check_rate(evaluation[key])

            if not math.isclose(
                evaluation[key], expected,
                rel_tol=0, abs_tol=1e-12,
            ):
                raise ValueError(f"Не совпала метрика: {key}")


def validate_classifier_grid(classifiers, prefix=""):
    expected = {
        f"{prefix}{method}_C{c_value}": (method, c_value)
        for method in ("logistic", "svm")
        for c_value in (0.01, 0.1, 1.0)
    }

    if set(classifiers) != set(expected):
        raise ValueError("Неожиданная сетка классификаторов")

    for key, (method, c_value) in expected.items():
        result = classifiers[key]
        validate_backend_evaluation(result)

        if result["method"] != method or result["C"] != c_value:
            raise ValueError("Не совпали параметры классификатора")

        limit = 2000 if method == "logistic" else 20000

        if (
            type(result["iterations"]) is not int
            or not 0 < result["iterations"] < limit
        ):
            raise ValueError("Некорректное число итераций")


def best_classifier_key(classifiers, method=None):
    candidates = [
        key for key, result in classifiers.items()
        if method is None or result["method"] == method
    ]

    if not candidates:
        raise ValueError("Нет подходящих вариантов")

    return min(
        candidates,
        key=lambda key: (
            classifiers[key]["balanced_error"],
            classifiers[key]["mean_far"],
            classifiers[key]["C"],
        ),
    )


def print_backend_table(results):
    print(
        f"{'Вариант':<27}{'Порог':>12}"
        f"{'FAR':>10}{'FRR':>10}{'Ошибка':>10}"
    )

    for name, result in results.items():
        far = float(np.mean([
            item["far"] for item in result["scenarios"].values()
        ]))
        frr = float(np.mean([
            item["frr"] for item in result["scenarios"].values()
        ]))

        print(
            f"{name:<27}{result['threshold']:>12.6f}"
            f"{far:>10.2%}{frr:>10.2%}"
            f"{result['balanced_error']:>10.2%}"
        )


def load_pair_classifier_comparison():
    report = read_verified_report("pair_classifiers")

    if Path(report["encoder_checkpoint"]).resolve() != Path(
        test_1000_summary["checkpoint"]
    ).resolve():
        raise ValueError("Указан другой encoder")

    validate_backend_evaluation(report["cosine"])
    validate_classifier_grid(report["classifiers"])

    if set(report["best_keys"]) != {"logistic", "svm"}:
        raise ValueError("Неожиданные выбранные методы")

    for method in ("logistic", "svm"):
        if report["best_keys"][method] != best_classifier_key(
            report["classifiers"], method
        ):
            raise ValueError("Не совпал лучший классификатор")

    return report


pair_classifier_summary = None
pair_classifier_summary = load_pair_classifier_comparison()

print("СОХРАНЁННЫЙ VALIDATION — без повторного обучения\n")
print_backend_table({
    "ECAPA + cosine": pair_classifier_summary["cosine"],
    **pair_classifier_summary["classifiers"],
})

print("\nЛучшие варианты:", pair_classifier_summary["best_keys"])
print("scikit-learn исходного запуска:",
      pair_classifier_summary["sklearn_version"])
print("Хеш, метрики и выбор вариантов проверены. Test не использовался.")

**Результат:** cosine — 2.10%, лучший logistic — 2.93%, лучший SVM — 3.66%.

**Решение:** оставили cosine. Обучаемые парные классификаторы не улучшили validation-качество новых дикторов.

**Ограничение:** ECAPA уже адаптировалась на train-дикторах, поэтому качество на train могло быть оптимистичным. Вывод относится к проверенным признакам, линейным классификаторам и сетке C.

## E22. ECAPA + PLDA

**Гипотеза:** статистическое моделирование междикторской и внутридикторской вариативности улучшит сравнение.

PLDA обучали на train-эмбеддингах через SpeechBrain: rank 32/64, 10 итераций, whiten=False, scaling_factor=1.0. После обучения проверили shrinkage остаточной ковариации 0 и 0.05. Вариант и порог выбрали на validation, затем проверили выбранный вариант на test.

In [ ]:
def regularize_plda_covariance(sigma, shrinkage):
    sigma = np.asarray(sigma)

    if (
        sigma.shape != (192, 192)
        or sigma.dtype.kind != "f"
        or not np.isfinite(sigma).all()
        or isinstance(shrinkage, bool)
        or shrinkage not in (0.0, 0.05)
    ):
        raise ValueError("Некорректная ковариация или shrinkage")

    symmetric = (
        sigma.astype(np.float64) + sigma.astype(np.float64).T
    ) / 2

    average_variance = float(np.trace(symmetric) / 192)

    regularized = (
        (1 - shrinkage) * symmetric
        + shrinkage * average_variance * np.eye(192)
    )

    eigenvalues = np.linalg.eigvalsh(regularized)

    if (
        not np.isfinite(regularized).all()
        or not np.isfinite(eigenvalues).all()
        or eigenvalues.min() <= 0
    ):
        raise ValueError("Ковариация не положительно определена")

    return regularized


print("Логика регуляризации сохранена.")
print("Обучение PLDA и scoring не выполнялись.")

In [ ]:
def load_plda_comparison():
    validation = read_verified_report("plda_validation")
    test = read_verified_report("plda_test")

    if Path(validation["encoder_checkpoint"]).resolve() != Path(
        test_1000_summary["checkpoint"]
    ).resolve():
        raise ValueError("Указан другой encoder")

    if type(validation["iterations"]) is not int or validation["iterations"] != 10:
        raise ValueError("Неожиданное число итераций PLDA")

    validate_backend_evaluation(validation["cosine"])

    expected = {
        f"plda_rank{rank}_shrink{shrinkage}": (rank, shrinkage)
        for rank in (32, 64)
        for shrinkage in (0.0, 0.05)
    }

    if set(validation["variants"]) != set(expected):
        raise ValueError("Неожиданная сетка PLDA")

    for key, (rank, shrinkage) in expected.items():
        result = validation["variants"][key]
        validate_backend_evaluation(result)

        eigenvalue = result["minimum_sigma_eigenvalue"]

        if (
            result["rank"] != rank
            or result["shrinkage"] != shrinkage
            or isinstance(eigenvalue, bool)
            or not isinstance(eigenvalue, (int, float))
            or not math.isfinite(eigenvalue)
            or eigenvalue <= 0
        ):
            raise ValueError("Некорректные параметры PLDA")

    best_key = min(
        validation["variants"],
        key=lambda key: (
            validation["variants"][key]["balanced_error"],
            validation["variants"][key]["mean_far"],
            validation["variants"][key]["rank"],
        ),
    )

    if (
        validation["best_key"] != best_key
        or test["variant"] != best_key
        or test["threshold"] != validation["variants"][best_key]["threshold"]
        or test["cosine_threshold"]
        != test_1000_summary["thresholds"]["trained"]
        or test["cosine_mean_error"]
        != test_1000_summary["mean_errors"]["trained"]
    ):
        raise ValueError("Не согласованы выбранный вариант и test")

    validate_backend_evaluation({
        "threshold": test["threshold"],
        "balanced_error": test["plda_mean_error"],
        "scenarios": test["plda_scenarios"],
    })

    return validation, test


plda_comparison = None
plda_comparison = load_plda_comparison()
plda_validation, plda_test = plda_comparison

print("СОХРАНЁННЫЙ VALIDATION\n")
print_backend_table({
    "ECAPA + cosine": plda_validation["cosine"],
    **plda_validation["variants"],
})

print("\nСОХРАНЁННЫЙ TEST — порог не менялся\n")
print(
    f"{'Сценарий':<16}{'Метод':<10}"
    f"{'FAR':>10}{'FRR':>10}{'Ошибка':>10}"
)

for scenario in ("clean-clean", "clean-noisy", "noisy-noisy"):
    for method, metrics in (
        ("cosine", test_1000_summary["scenarios"]["trained"][scenario]),
        ("PLDA", plda_test["plda_scenarios"][scenario]),
    ):
        print(
            f"{scenario:<16}{method:<10}"
            f"{metrics['far']:>10.2%}{metrics['frr']:>10.2%}"
            f"{metrics['balanced_error']:>10.2%}"
        )

print(f"\nCosine: {plda_test['cosine_mean_error']:.2%}")
print(f"PLDA:   {plda_test['plda_mean_error']:.2%}")
print("Хеши, метрики и выбор по validation проверены.")
print("Матрицы PLDA и их собственные значения заново не рассчитывались.")

**Результат:** лучший PLDA дал 1.44% на validation против 2.10% у cosine. На test PLDA получил 3.44% против 3.07%; особенно ухудшился clean-noisy.

**Решение:** PLDA не включили в основной вариант. Validation-выигрыш не перенёсся на test.

**Ограничение:** после просмотра test не продолжаем подбирать PLDA по этому же test. Результат относится к данной реализации, объёму train и проверенной регуляризации.

## E23. MFCC + линейный классификатор

**Гипотеза:** простые акустические статистики могут служить независимым классическим baseline.

Из 20 MFCC вычисляли среднее и стандартное отклонение: 40 признаков записи. Симметричные признаки пары имели размерность 80. Использовали те же пары, что в ECAPA-эксперименте, и ту же сетку logistic/SVM. Scaler обучали только на train.

In [ ]:
def mfcc_statistics_for_experiment(signal, sample_rate=8000):
    import librosa

    signal = np.asarray(signal)

    if (
        sample_rate != 8000
        or signal.ndim != 1
        or signal.dtype != np.dtype("float32")
        or not 24000 <= signal.size <= 40000
        or not np.isfinite(signal).all()
    ):
        raise ValueError("Нужен конечный float32 mono-сигнал 8 кГц, 3–5 с")

    coefficients = librosa.feature.mfcc(
        y=signal.copy(),
        sr=sample_rate,
        n_mfcc=20,
        n_mels=40,
        n_fft=256,
        win_length=200,
        hop_length=80,
        fmin=50,
        fmax=3800,
        center=False,
        dct_type=2,
        norm="ortho",
    )

    features = np.concatenate([
        coefficients.mean(axis=1),
        coefficients.std(axis=1),
    ]).astype(np.float32)

    if features.shape != (40,) or not np.isfinite(features).all():
        raise RuntimeError("Некорректные MFCC-признаки")

    return features


print("Извлечение MFCC-статистик сохранено, но не запускалось.")
print("Для пары используется symmetric_pair_features из E21.")
print("Для классификаторов — make_pair_classifier из E21.")

In [ ]:
def load_mfcc_comparison():
    report = read_verified_report("mfcc")

    expected_config = {
        "sample_rate": 8000,
        "max_seconds": 5,
        "n_mfcc": 20,
        "n_mels": 40,
        "n_fft": 256,
        "win_length": 200,
        "hop_length": 80,
        "fmin": 50,
        "fmax": 3800,
        "center": False,
        "statistics": ["mean", "std"],
    }

    if report["feature_config"] != expected_config:
        raise ValueError("Неожиданная конфигурация MFCC")

    validate_classifier_grid(report["classifiers"], prefix="mfcc_")
    validate_backend_evaluation(report["ecapa_cosine"])

    if report["best_key"] != best_classifier_key(report["classifiers"]):
        raise ValueError("Не совпал лучший MFCC-вариант")

    return report


mfcc_summary = None
mfcc_summary = load_mfcc_comparison()

print("СОХРАНЁННЫЙ VALIDATION — без извлечения признаков и обучения\n")
print_backend_table({
    "ECAPA + cosine": mfcc_summary["ecapa_cosine"],
    **mfcc_summary["classifiers"],
})

best_mfcc = mfcc_summary["classifiers"][mfcc_summary["best_key"]]

print("\nЛучший вариант:", mfcc_summary["best_key"])
print(f"Ошибка MFCC: {best_mfcc['balanced_error']:.2%}")
print(
    "Ошибка ECAPA + cosine: "
    f"{mfcc_summary['ecapa_cosine']['balanced_error']:.2%}"
)
print("Хеш, конфигурация, метрики и выбор варианта проверены.")
print("Идентичность исходных пар здесь повторно не проверялась.")
print("Test не использовался.")

**Результат:** лучший MFCC-вариант получил 24.95% против 2.10% у ECAPA + cosine.

**Решение:** сохранили как классический контрольный эксперимент, но не включили в основной конвейер.

**Ограничение:** проверены простые статистики MFCC, теряющие временную структуру. Результат не относится ко всем MFCC-системам: полноценный i-vector pipeline проверили отдельно в E24.

## E24. Kaldi i-vector + PLDA

**Гипотеза:** готовый классический pipeline может конкурировать с ECAPA.

На Ubuntu-сервере использовали Kaldi 0007_voxceleb_v1_1a и поставляемый с ним PLDA. Конвейер: MFCC → VAD → deltas → sliding CMN → voiced frames → UBM posteriors → i-vector 400 → центрирование → LDA → length normalization → PLDA.

Обработали 102 validation-записи и 5151 пару. Порог выбрали на validation. Контейнер запускали без сети, от непривилегированного пользователя, с ограничениями CPU/RAM и read-only корневой файловой системой. Рабочий каталог был доступен для записи.

In [ ]:
def load_ivector_comparison():
    report = read_verified_report("ivector")

    expected_container = (
        "kaldiasr/kaldi@sha256:"
        "335fa60ff1b70d5145dfea83bb6e4cd7b9b8e40bfbf11b8688cd04b358f952f2"
    )

    if (
        report["model"] != "Kaldi 0007_voxceleb_v1_1a"
        or report["backend"] != "bundled PLDA"
        or report["container"] != expected_container
    ):
        raise ValueError("Неожиданная модель, backend или контейнер")

    validate_backend_evaluation(report["ivector"])
    validate_backend_evaluation(report["ecapa_cosine"])

    if not math.isclose(
        report["ecapa_cosine"]["balanced_error"],
        pair_classifier_summary["cosine"]["balanced_error"],
        rel_tol=0,
        abs_tol=1e-12,
    ):
        raise ValueError("Не совпал ECAPA-ориентир")

    return report


ivector_summary = None
ivector_summary = load_ivector_comparison()

print("СОХРАНЁННЫЙ VALIDATION — Kaldi повторно не запускался\n")
print("Модель:", ivector_summary["model"])
print("Контейнер:", ivector_summary["container"])
print(f"Порог i-vector + PLDA: {ivector_summary['ivector']['threshold']:.6f}\n")

print(
    f"{'Сценарий':<16}{'FAR':>10}{'FRR':>10}{'Ошибка':>10}"
)

for scenario in ("clean-clean", "clean-noisy", "noisy-noisy"):
    metrics = ivector_summary["ivector"]["scenarios"][scenario]

    print(
        f"{scenario:<16}"
        f"{metrics['far']:>10.2%}"
        f"{metrics['frr']:>10.2%}"
        f"{metrics['balanced_error']:>10.2%}"
    )

print(
    "\nECAPA + cosine:",
    f"{ivector_summary['ecapa_cosine']['balanced_error']:.2%}",
)
print(
    "i-vector + PLDA:",
    f"{ivector_summary['ivector']['balanced_error']:.2%}",
)
print("\nХеш отчёта, конфигурация и формулы метрик проверены.")
print("Все исходные i-vector и оценки пар здесь заново не проверялись.")
print("Test для этого варианта не использовался.")

**Результат:** i-vector + PLDA получил 14.53% против 2.10% у ECAPA + cosine.

**Решение:** эксперимент закрыли, Kaldi не включили в основной конвейер.

**Ограничение:** проверили конкретный готовый pipeline, а не все i-vector-системы. Собственный extractor на наших обучающих данных не обучали. Закрепление digest контейнера фиксирует его содержимое, но само по себе не доказывает отсутствие уязвимостей.

## E25. Итоговый прототип

Выбрали ECAPA после адаптации на Vibravox с classification loss, checkpoint шага 500. Используем cosine similarity, автоматический выбор порога по цене ошибок и отдельный калибратор confidence.

Основной конвейер не содержит шумоподавления, удаления пауз и дополнительного усиления. Рабочий интерфейс, проверки WAV и загрузка модели находятся в notebook 01. Этот notebook сохраняет экспериментальную историю и анализ результатов.

In [ ]:
def summarize_final_decision():
    training = extended_training_summary["classification"]
    calibration = calibration_summary

    if (
        training["best_step"] != 500
        or calibration["step"] != 500
        or training["best_validation"]["threshold"]
        != calibration["decision_threshold"]
        or test_1000_summary["thresholds"]["trained"]
        != calibration["decision_threshold"]
    ):
        raise ValueError("Не согласованы итоговая модель и настройки")

    return {
        "encoder": "ECAPA-TDNN",
        "adaptation": "Vibravox, classification, fc only",
        "checkpoint_step": training["best_step"],
        "comparison": "cosine similarity",
        "reference_validation_threshold": calibration["decision_threshold"],
        "decision_policy": "minimum validation expected error cost",
        "default_false_accept_cost": 1.0,
        "default_false_reject_cost": 1.0,
        "assumed_same_prior": 0.5,
        "scenario_weights": "equal",
        "denoising": False,
        "pause_removal": False,
        "extra_volume_normalization": False,
        "calibration_a": calibration["a"],
        "calibration_b": calibration["b"],
    }


final_decision = None
final_decision = summarize_final_decision()

print("ИТОГОВОЕ РЕШЕНИЕ\n")

for key, value in final_decision.items():
    print(f"{key}: {value}")

print("\nОбщая test-ошибка:")
print(f"baseline: {test_1000_summary['mean_errors']['baseline']:.2%}")
print(f"trained:  {test_1000_summary['mean_errors']['trained']:.2%}")

overall_interval = speaker_bootstrap_summary["results"]["mean"]

print(
    "95% интервал изменения, п.п.: "
    f"[{overall_interval['ci95_lower_pp']:+.3f}; "
    f"{overall_interval['ci95_upper_pp']:+.3f}]"
)

print("\nreference_validation_threshold — исторический ориентир.")
print("В notebook 01 рабочий порог выбирается по стоимости ошибок")
print("и может немного отличаться из-за положения между соседними score.")

**Почему выбран этот вариант:** обработка аудио и дополнительные backend не дали убедительного преимущества. Адаптация ECAPA улучшила целевой clean-noisy сценарий, но ухудшила noisy-noisy.

**Что не утверждаем:** общего статистически подтверждённого превосходства trained над baseline нет. Confidence предварительный; работа на короткой речи и реальном треске требует отдельного подтверждения.

**Перед сдачей:** выполнить Restart Kernel → Run All в обоих notebook и проверить выбранный checkpoint на треске в первой, второй и обеих записях. Желательна окончательная оценка на новых отложенных дикторах.

## Проверки безопасности и границы гарантий

В этом notebook отчёты читаются по разрешённому списку, с SHA-256, ограничением размера и строгим разбором JSON. Не запускаются shell-команды, скачивание, обучение или загрузка сериализованных моделей.

Ниже проверяем отклонение некорректных данных и подменённого содержимого отчёта. Подмена выполняется только в памяти через mock; реальные файлы не изменяются. Это ограниченные регрессионные проверки, а не доказательство полной безопасности приложения.

In [ ]:
def run_notebook_safety_checks():
    from unittest.mock import patch

    passed = []

    def expect_value_error(name, action):
        try:
            action()
        except ValueError:
            passed.append(name)
        else:
            raise RuntimeError(f"Проверка не прошла: {name}")

    def parse_test_json(text):
        result = json.loads(
            text,
            parse_constant=reject_json_constant,
            parse_float=finite_json_float,
            object_pairs_hook=unique_json_keys,
        )

        if not isinstance(result, dict):
            raise ValueError("Ожидался JSON-объект")

        return result

    expect_value_error(
        "Запрет неразрешённого имени отчёта",
        lambda: read_verified_report("../../unknown.json"),
    )

    invalid_json_cases = (
        ("Повторные ключи", '{"x": 1, "x": 2}'),
        ("NaN", '{"x": NaN}'),
        ("Infinity", '{"x": Infinity}'),
        ("Переполнение float", '{"x": 1e999}'),
        ("Некорректный корневой тип", "[]"),
    )

    for name, text in invalid_json_cases:
        expect_value_error(
            f"JSON: {name}",
            lambda text=text: parse_test_json(text),
        )

    # Подменяем только чтение, не содержимое настоящего файла.
    with patch.object(
        Path,
        "open",
        return_value=io.BytesIO(b'{"tampered": true}'),
    ):
        expect_value_error(
            "Отклонение отчёта с неверным SHA-256",
            lambda: read_verified_report("wavlm"),
        )

    for invalid_rate in (True, -0.1, float("nan")):
        expect_value_error(
            f"Отклонение некорректной доли: {invalid_rate}",
            lambda value=invalid_rate: check_rate(value),
        )

    expect_value_error(
        "Отклонение нормализации тишины",
        lambda: normalize_volume_for_experiment(
            np.zeros(8000, dtype=np.float32)
        ),
    )

    pair = {
        "first_id": "record_a",
        "second_id": "record_b",
        "is_same": True,
        "similarity": 0.1,
    }
    reverse_pair = {
        **pair,
        "first_id": "record_b",
        "second_id": "record_a",
    }

    forward = analyze_same_pair_errors([pair], 0.2)
    reverse = analyze_same_pair_errors([reverse_pair], 0.2)

    if forward != reverse:
        raise RuntimeError("Нарушена симметрия анализа ошибок")

    passed.append("Симметрия анализа ошибочной пары")

    expect_value_error(
        "Отклонение повторной пары в обратном порядке",
        lambda: analyze_same_pair_errors([pair, reverse_pair], 0.2),
    )

    # В конце заново проверяем реальные JSON-отчёты без mock.
    reports = load_experiment_reports()

    if len(reports) != 16:
        raise RuntimeError("Неожиданное число отчётов")

    passed.append("Повторная проверка 16 реальных JSON-отчётов")

    return passed


notebook_safety_checks = None
notebook_safety_checks = run_notebook_safety_checks()

for name in notebook_safety_checks:
    print("✓", name)

print("\nПройдено проверок:", len(notebook_safety_checks))
print("Реальные файлы не изменялись.")
print("Аудиодекодер, GPU, Gradio и инфраструктура здесь не тестировались.")

## Что защищено и что остаётся

**В notebook 01 реализованы:** проверка комплекта артефактов по хешам, ограниченная загрузка checkpoint, проверка структуры весов, ограниченное чтение WAV, проверка конечности сигналов и эмбеддингов, последовательный доступ к модели и локальный Gradio.

**В notebook 02 реализованы:** проверенное чтение отчётов, валидация метрик, сохранение экспериментальной логики без автоматического запуска тяжёлых операций и явное обозначение исторических результатов.

**В инфраструктуре предстоит обеспечить:** защиту доверенного манифеста, запрет записи сервисом в каталог модели, контроль доступа, ограничения загрузок и ресурсов, изоляцию декодирования, журналирование, мониторинг и восстановление.

Хеши не защищают, если одновременно подменены файлы и ожидаемые хеши. Валидация отчёта не доказывает достоверность измерений. Проверки этого notebook не заменяют аудит зависимостей и тестирование сервиса.